In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Ready!")

Mounted at /content/drive
Ready!


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ============================================================
# SECTION 1 — LOAD ALL DATA
# ============================================================
print("=" * 55)
print("SECTION 1 — LOADING DATA")
print("=" * 55)

data_path = '/content/drive/MyDrive/player_ratings/data'

all_seasons = []
for file in sorted(os.listdir(data_path)):
    if file.endswith('_PL.csv') and not file.startswith('GK'):
        season = int(file.split('_')[0])
        df = pd.read_csv(f'{data_path}/{file}')
        df['season'] = season
        all_seasons.append(df)
data = pd.concat(all_seasons, ignore_index=True)

gk_seasons = []
for file in sorted(os.listdir(data_path)):
    if file.startswith('GK_') and file.endswith('_PL.csv'):
        season = int(file.split('_')[1])
        df = pd.read_csv(f'{data_path}/{file}')
        df['season'] = season
        gk_seasons.append(df)
gk_data = pd.concat(gk_seasons, ignore_index=True)

injuries = pd.read_csv(f'{data_path}/injuries_combined.csv')
players_df = pd.read_csv(f'{data_path}/players_combined_complete.csv')

print(f"Outfield: {len(data)} rows")
print(f"GK: {len(gk_data)} rows")
print(f"Injuries: {len(injuries)} records")
print(f"Players: {len(players_df)} records")

# ============================================================
# SECTION 2 — CLEAN DATA
# ============================================================
print("\n" + "=" * 55)
print("SECTION 2 — CLEANING DATA")
print("=" * 55)

data = data[data['ninety_mins_played'] >= 5].copy()
gk_data = gk_data[gk_data['ninety_mins_played'] >= 5].copy()

def simplify_position(pos):
    if pd.isna(pos): return 'Unknown'
    pos = str(pos).upper()
    if 'GK' in pos: return 'GK'
    elif pos.startswith('DF') or pos in ['DF,MF', 'MF,DF']: return 'DF'
    elif pos.startswith('MF') or pos == 'MF,FW': return 'MF'
    elif pos.startswith('FW') or pos == 'FW,MF': return 'FW'
    else: return 'Unknown'

data['position_group'] = data['position'].apply(simplify_position)
data = data[data['position_group'] != 'Unknown'].copy()

counting_stats = ['interceptions', 'clearances', 'progressive_passes',
                  'key_passes', 'tackles_interceptions', 'ball_recoveries',
                  'shots', 'goals', 'assists']
for col in counting_stats:
    if col in data.columns:
        data[f'{col}_per90'] = (data[col] / data['ninety_mins_played']).round(3)

data = data.sort_values(['player', 'season']).reset_index(drop=True)
gk_data = gk_data.sort_values(['player', 'season']).reset_index(drop=True)

print(f"Outfield rows: {len(data)}")
print(f"GK rows: {len(gk_data)}")
print("\nData quality check:")
for season in sorted(data['season'].unique()):
    max_mins = data[data['season'] == season]['ninety_mins_played'].max()
    flag = '⚠️' if max_mins < 30 else '✓'
    print(f"  {season}: {flag} max={max_mins:.0f} x90s")

# ============================================================
# SECTION 3 — INJURY + MARKET VALUE FEATURES
# ============================================================
print("\n" + "=" * 55)
print("SECTION 3 — INJURY + MARKET VALUE FEATURES")
print("=" * 55)

def convert_season_inj(s):
    try:
        end = int(str(s).split('/')[1])
        return 2000 + end if end < 50 else 1900 + end
    except: return None

injuries['season_year'] = injuries['season'].apply(convert_season_inj)
injuries_pl = injuries[injuries['season_year'] >= 2017].copy()
injuries_pl['player_name_clean'] = injuries_pl['player_name'].str.strip().str.lower()

def parse_mv(val_str):
    try:
        v = str(val_str).strip().lower()
        if 'm' in v: return float(v.replace('m','').strip())
        elif 'k' in v: return float(v.replace('k','').strip()) / 1000
        else: return float(v)
    except: return np.nan

players_df['market_value_m'] = players_df['market_value'].apply(parse_mv)
players_df['player_name_clean'] = players_df['player_name'].str.strip().str.lower()

def convert_season_pl(s):
    try:
        end = int(str(s).split('-')[1])
        return 2000 + end if end < 50 else 1900 + end
    except: return None

players_df['season_year'] = players_df['season'].apply(convert_season_pl)

def get_injury_features(player_name, season_year):
    nc = str(player_name).strip().lower()
    pi = injuries_pl[injuries_pl['player_name_clean'] == nc]
    f = {'games_missed_last_season': 0, 'injuries_last_12m': 0,
         'career_injury_count': 0, 'hamstring_history': 0,
         'same_bodypart_before': 0, 'returned_from_long_injury': 0}
    if len(pi) == 0: return f
    f['career_injury_count'] = len(pi)
    ls = pi[pi['season_year'] == season_year - 1]
    f['games_missed_last_season'] = ls['games_missed'].sum()
    recent = pi[pi['season_year'].isin([season_year-1, season_year])]
    f['injuries_last_12m'] = len(recent)
    ham = pi[pi['body_part'].str.lower().str.contains('hamstring|thigh', na=False)]
    f['hamstring_history'] = 1 if len(ham) > 0 else 0
    li = pi[(pi['season_year'] >= season_year-2) & (pi['days_out'] >= 90)]
    f['returned_from_long_injury'] = 1 if len(li) > 0 else 0
    if len(pi) >= 2:
        bp = pi['body_part'].dropna().str.lower().values
        f['same_bodypart_before'] = 1 if len(set(bp)) < len(bp) else 0
    return f

injury_cols = ['games_missed_last_season', 'injuries_last_12m',
               'career_injury_count', 'hamstring_history',
               'same_bodypart_before', 'returned_from_long_injury']
for col in injury_cols + ['market_value_m']:
    data[col] = np.nan

for idx, row in data.iterrows():
    player = row['player']
    season = row['season']
    inj = get_injury_features(player, season)
    for col, val in inj.items():
        data.at[idx, col] = val
    nc = str(player).strip().lower()
    pv = players_df[players_df['player_name_clean'] == nc]
    if len(pv) > 0:
        sm = pv[pv['season_year'] == season]
        if len(sm) > 0:
            data.at[idx, 'market_value_m'] = sm.iloc[0]['market_value_m']
        else:
            cl = pv.iloc[(pv['season_year'] - season).abs().argsort()[:1]]
            data.at[idx, 'market_value_m'] = cl.iloc[0]['market_value_m']

print(f"Injury coverage: {data['games_missed_last_season'].notna().sum()}/{len(data)}")
print(f"Market value coverage: {data['market_value_m'].notna().sum()}/{len(data)}")

# ============================================================
# SECTION 4 — LEAK-FREE SUB-POSITION CLUSTERING
# ============================================================
print("\n" + "=" * 55)
print("SECTION 4 — LEAK-FREE SUB-POSITION CLUSTERING")
print("=" * 55)

fw_feats = ['xg_per_90', 'take_on_succ_pct', 'aerials_won_pct',
            'shots_per_90', 'assists_per_90', 'progressive_carries']
df_feats = [f for f in ['clearances_per90', 'aerials_won_pct',
            'progressive_passes_per90', 'interceptions_per90',
            'tackles_interceptions_per90'] if f in data.columns]
mf_feats = ['key_passes_per90', 'assists_per_90', 'tackles_interceptions_per90',
            'progressive_passes_per90', 'xg_per_90', 'ball_recoveries_per90']

pos_feats = {'FW': fw_feats, 'DF': df_feats, 'MF': mf_feats}
pos_clusters = {'FW': 2, 'DF': 2, 'MF': 3}

data_pre2024 = data[data['season'] < 2024].copy()
data_2024 = data[data['season'] == 2024].copy()
data_pre2024['subposition'] = data_pre2024['position_group']
data_2024['subposition'] = data_2024['position_group']

for position in ['FW', 'DF', 'MF']:
    feats = pos_feats[position]
    n = pos_clusters[position]
    pos_pre = data_pre2024[data_pre2024['position_group'] == position]
    valid_pre = pos_pre[feats].dropna()
    scaler = StandardScaler()
    X_pre = scaler.fit_transform(valid_pre)
    km = KMeans(n_clusters=n, random_state=42, n_init=10)
    km.fit(X_pre)
    c_pre = km.predict(X_pre)
    data_pre2024.loc[valid_pre.index, 'cluster_raw'] = c_pre
    pos_24 = data_2024[data_2024['position_group'] == position]
    valid_24 = pos_24[feats].dropna()
    if len(valid_24) > 0:
        c_24 = km.predict(scaler.transform(valid_24))
        data_2024.loc[valid_24.index, 'cluster_raw'] = c_24
    pwc = valid_pre.copy()
    pwc['cluster'] = c_pre
    if position == 'FW':
        xgm = pwc.groupby('cluster')['xg_per_90'].mean()
        st_c, w_c = int(xgm.idxmax()), int(xgm.idxmin())
        for dp in [data_pre2024, data_2024]:
            pp = dp[dp['position_group'] == position]
            dp.loc[pp[pp['cluster_raw'] == st_c].index, 'subposition'] = 'ST'
            dp.loc[pp[pp['cluster_raw'] == w_c].index, 'subposition'] = 'W'
    elif position == 'DF':
        clm = pwc.groupby('cluster')['clearances_per90'].mean()
        cb_c, fb_c = int(clm.idxmax()), int(clm.idxmin())
        for dp in [data_pre2024, data_2024]:
            pp = dp[dp['position_group'] == position]
            dp.loc[pp[pp['cluster_raw'] == cb_c].index, 'subposition'] = 'CB'
            dp.loc[pp[pp['cluster_raw'] == fb_c].index, 'subposition'] = 'FB'
    elif position == 'MF':
        kpm = pwc.groupby('cluster')['key_passes_per90'].mean()
        sc = kpm.sort_values()
        dm_c, cm_c, am_c = int(sc.index[0]), int(sc.index[1]), int(sc.index[2])
        lmap = {dm_c: 'DM', cm_c: 'CM', am_c: 'AM'}
        for dp in [data_pre2024, data_2024]:
            pp = dp[dp['position_group'] == position]
            for cid, lab in lmap.items():
                dp.loc[pp[pp['cluster_raw'] == cid].index, 'subposition'] = lab

data_clean = pd.concat([data_pre2024, data_2024], ignore_index=True)
data_clean.loc[data_clean['position_group'] == 'GK', 'subposition'] = 'GK'

overrides = {'Antonee Robinson': 'FB', 'John Mcginn': 'CM',
             'Mario Lemina': 'DM', 'Lewis Cook': 'DM',
             'Keane Lewis-Potter': 'W', 'Ismaila Sarr': 'W'}
for player, sp in overrides.items():
    data_clean.loc[data_clean['player'] == player, 'subposition'] = sp

print("Sub-position counts:")
print(data_clean['subposition'].value_counts())

# ============================================================
# SECTION 5 — FEATURES, TARGETS, PARAMS
# ============================================================
base_features = [
    'age', 'ninety_mins_played', 'goals_per_90', 'assists_per_90',
    'xg_per_90', 'xag_per_90', 'shots_per_90', 'shots_on_target_pct',
    'take_on_succ_pct', 'pass_completion_pct', 'progressive_passes_per90',
    'tackles_interceptions_per90', 'ball_recoveries_per90',
    'key_passes_per90', 'interceptions_per90', 'clearances_per90',
    'aerials_won_pct', 'tackle_pct',
    'games_missed_last_season', 'injuries_last_12m',
    'career_injury_count', 'hamstring_history',
    'same_bodypart_before', 'returned_from_long_injury',
    'market_value_m',
]

position_targets_fixed = {
    'FW': ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'DF': ['aerials_won_pct', 'pass_completion_pct',
           'interceptions_per90', 'clearances_per90'],
    'MF': ['pass_completion_pct', 'progressive_passes_per90',
           'key_passes_per90', 'tackles_interceptions_per90',
           'ball_recoveries_per90', 'assists_per_90']
}

pct_stats_final = ['aerials_won_pct', 'pass_completion_pct']

best_params_v2 = {
    'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01,
    'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 8,
    'reg_alpha': 0.1, 'reg_lambda': 2.0, 'random_state': 42, 'verbosity': 0
}

subpos_to_pos = {'ST': 'FW', 'W': 'FW', 'CB': 'DF', 'FB': 'DF',
                 'DM': 'MF', 'CM': 'MF', 'AM': 'MF'}

subpos_targets_final = {
    'ST': ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'W':  ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'CB': ['aerials_won_pct', 'pass_completion_pct',
           'interceptions_per90', 'clearances_per90'],
    'FB': ['pass_completion_pct', 'interceptions_per90',
           'clearances_per90', 'assists_per_90'],
    'DM': ['tackles_interceptions_per90', 'ball_recoveries_per90',
           'pass_completion_pct'],
    'CM': ['progressive_passes_per90', 'tackles_interceptions_per90',
           'key_passes_per90', 'pass_completion_pct'],
    'AM': ['assists_per_90', 'key_passes_per90',
           'progressive_passes_per90', 'goals_per_90'],
}

cv_mae_per_metric = {
    'goals_per_90': 2.32, 'assists_per_90': 1.64, 'xg_per_90': 1.69,
    'shots_per_90': 7.44, 'aerials_won_pct': 8.07, 'pass_completion_pct': 3.29,
    'interceptions_per90': 5.04, 'clearances_per90': 10.91,
    'progressive_passes_per90': 18.49, 'key_passes_per90': 6.88,
    'tackles_interceptions_per90': 11.40, 'ball_recoveries_per90': 16.73,
}

print(f"Features: {len(base_features)}, Targets and params defined!")

# ============================================================
# SECTION 6 — HELPER FUNCTIONS
# ============================================================
def create_multi_season_features(df, player, current_season):
    all_data = df[(df['player'] == player) &
                  (df['season'] < current_season)].sort_values('season')
    if len(all_data) == 0:
        return None
    recent = all_data.tail(3)
    features = {
        'seasons_available': len(recent),
        'total_pl_seasons': len(all_data)
    }
    for feat in base_features:
        if feat not in all_data.columns:
            continue
        av = all_data[feat].values
        rv = recent[feat].values
        features[f'{feat}_s1'] = rv[-1] if len(rv) >= 1 else np.nan
        features[f'{feat}_s2'] = rv[-2] if len(rv) >= 2 else np.nan
        features[f'{feat}_s3'] = rv[-3] if len(rv) >= 3 else np.nan
        features[f'{feat}_trend'] = (rv[-1] - rv[-2]) if len(rv) >= 2 else np.nan
        features[f'{feat}_avg'] = np.nanmean(av)
        vv = av[~np.isnan(av.astype(float))]
        features[f'{feat}_longterm_trend'] = (vv[-1] - vv[0]) if len(vv) >= 2 else np.nan
        features[f'{feat}_career_best'] = np.nanmax(av) if len(av) > 0 else np.nan
    features['age'] = all_data['age'].values[-1]
    features['last_season_minutes'] = all_data['ninety_mins_played'].values[-1]
    return features

def build_dataset(df, position, up_to_season):
    rows = []
    pos_df = df[(df['position_group'] == position) &
                (df['season'] <= up_to_season)].copy()
    for player in pos_df['player'].unique():
        for season in sorted(pos_df[pos_df['player'] == player]['season'].unique()):
            features = create_multi_season_features(pos_df, player, season)
            if features is None:
                continue
            actual = pos_df[(pos_df['player'] == player) &
                           (pos_df['season'] == season)]
            if len(actual) == 0:
                continue
            actual = actual.iloc[0]
            for target in position_targets_fixed[position]:
                if target in actual:
                    features[f'target_{target}'] = actual[target]
            features['target_ninety_mins'] = actual['ninety_mins_played']
            features['player'] = player
            features['season'] = season
            features['position_group'] = position
            rows.append(features)
    return pd.DataFrame(rows)

def predict_minutes_v3(player, position, features, model_info, data_df):
    hist = data_df[(data_df['player'] == player) & (data_df['season'] < 2025)]
    if len(hist) == 0:
        return 15.0
    rm = hist.sort_values('season').tail(3)['ninety_mins_played'].values
    if len(rm) >= 3:
        wa = rm[-1]*0.70 + rm[-2]*0.20 + rm[-3]*0.10
    elif len(rm) == 2:
        wa = rm[-1]*0.75 + rm[-2]*0.25
    else:
        wa = rm[-1]*0.90
    fr = pd.DataFrame([features])
    for col in model_info['feature_cols']:
        if col not in fr.columns:
            fr[col] = 0
    fr = fr[model_info['feature_cols']].fillna(0)
    mp = float(model_info['model'].predict(fr)[0])
    blended = wa*0.70 + mp*0.30
    gm = features.get('games_missed_last_season_s1', 0) or 0
    i12 = features.get('injuries_last_12m_s1', 0) or 0
    if gm > 10 or i12 >= 3: blended *= 0.82
    elif gm > 5 or i12 >= 2: blended *= 0.90
    elif gm > 2: blended *= 0.95
    pmax = {'FW': 35, 'MF': 36, 'DF': 37, 'GK': 38}
    return round(max(3, min(pmax.get(position, 35), blended)), 1)

def get_confidence_interval(target, pred_val, pred_90s):
    mae = cv_mae_per_metric.get(target, 2.0)
    margin = round(mae * 1.5, 1)
    if target in pct_stats_final:
        return round(max(0, pred_val - margin), 1), round(pred_val + margin, 1)
    else:
        total = pred_val * pred_90s
        return round(max(0, total - margin), 1), round(total + margin, 1)

def get_trend_explanation(df, player, subposition):
    ph = df[(df['player'] == player) & (df['season'] <= 2024)].tail(3)
    if len(ph) < 2:
        return "Limited data available for trend analysis."
    age = ph['age'].values[-1]
    if pd.isna(age):
        return "Limited data available for trend analysis."
    age = float(age)
    exp = []
    if age <= 23: exp.append(f"At {int(age)}, still in development years with room to grow.")
    elif age <= 27: exp.append(f"At {int(age)}, entering peak years.")
    elif age <= 30: exp.append(f"At {int(age)}, in prime years but may begin to slow.")
    else: exp.append(f"At {int(age)}, experience is key but decline is possible.")
    if subposition in ['ST', 'W'] and 'goals_per_90' in ph.columns:
        v = ph['goals_per_90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.05: exp.append("Goal rate improving season on season. ↑")
            elif t < -0.05: exp.append("Goal rate has declined recently. ↓")
            else: exp.append("Goal rate has been consistent. →")
    elif subposition in ['CM', 'AM'] and 'progressive_passes_per90' in ph.columns:
        v = ph['progressive_passes_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.5: exp.append("Progressive passing improving. ↑")
            elif t < -0.5: exp.append("Progressive passing declining. ↓")
            else: exp.append("Consistent in progressing play. →")
    elif subposition == 'DM' and 'tackles_interceptions_per90' in ph.columns:
        v = ph['tackles_interceptions_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive contribution improving. ↑")
            elif t < -0.3: exp.append("Defensive contribution declining. ↓")
            else: exp.append("Defensively consistent. →")
    elif subposition in ['CB', 'FB'] and 'clearances_per90' in ph.columns:
        v = ph['clearances_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive output improving. ↑")
            elif t < -0.3: exp.append("Defensive output declining. ↓")
            else: exp.append("Defensively consistent. →")
    mv = ph['ninety_mins_played'].dropna().values
    if len(mv) >= 2:
        if mv[-1] - mv[-2] > 5: exp.append("Playing more minutes each season.")
        elif mv[-1] - mv[-2] < -5: exp.append("Minutes dropped recently — injury or form concern.")
    return " ".join(exp)

print("All helper functions defined!")

# ============================================================
# SECTION 7 — BUILD DATASETS
# ============================================================
print("\n" + "=" * 55)
print("SECTION 7 — BUILDING DATASETS")
print("=" * 55)

fw_full = build_dataset(data_clean, 'FW', up_to_season=2023)
df_full = build_dataset(data_clean, 'DF', up_to_season=2023)
mf_full = build_dataset(data_clean, 'MF', up_to_season=2023)

print(f"FW: {len(fw_full)} rows")
print(f"DF: {len(df_full)} rows")
print(f"MF: {len(mf_full)} rows")

# ============================================================
# SECTION 8 — TRAIN MODELS
# ============================================================
print("\n" + "=" * 55)
print("SECTION 8 — TRAINING MODELS")
print("=" * 55)

final_models_v2 = {}
feature_cols_v2 = [col for col in fw_full.columns
                   if col not in ['player', 'season', 'position_group']
                   and not col.startswith('target_')]

for position, dataset in [('FW', fw_full), ('DF', df_full), ('MF', mf_full)]:
    final_models_v2[position] = {}
    print(f"\n--- {position} ---")
    for target in position_targets_fixed[position] + ['ninety_mins']:
        target_col = f'target_{target}'
        if target_col not in dataset.columns:
            continue
        valid_cols = [col for col in feature_cols_v2 if col in dataset.columns]
        subset = dataset[valid_cols + [target_col]].dropna(subset=[target_col])
        if len(subset) < 10:
            continue
        X = subset[valid_cols].fillna(0)
        y = subset[target_col]
        model = XGBRegressor(**best_params_v2)
        model.fit(X, y)
        mae = mean_absolute_error(y, model.predict(X))
        final_models_v2[position][target] = {'model': model, 'feature_cols': valid_cols}
        print(f"  {target}: trained ✓  (train MAE: {mae:.3f})")

print("\nAll models trained!")

# ============================================================
# SECTION 9 — TEST ON 2024
# ============================================================
print("\n" + "=" * 55)
print("SECTION 9 — TESTING ON 2024")
print("=" * 55)

actual_2024 = data_clean[data_clean['season'] == 2024].copy()
fw_2024 = actual_2024[actual_2024['position_group'] == 'FW']
fw_hist = data_clean[(data_clean['position_group'] == 'FW') &
                     (data_clean['season'] < 2024)]

ge, ae, ag, aa, pg, pa = [], [], [], [], [], []

for player in fw_2024['player'].unique():
    features = create_multi_season_features(fw_hist, player, 2024)
    if features is None: continue
    ar = fw_2024[fw_2024['player'] == player]
    if len(ar) == 0: continue
    ar = ar.iloc[0]
    a90 = ar['ninety_mins_played']
    for target, errs, acts, preds in [
        ('goals_per_90', ge, ag, pg),
        ('assists_per_90', ae, aa, pa)]:
        if target not in final_models_v2['FW']: continue
        mi = final_models_v2['FW'][target]
        fr = pd.DataFrame([features])
        for col in mi['feature_cols']:
            if col not in fr.columns: fr[col] = 0
        fr = fr[mi['feature_cols']].fillna(0)
        pv = max(0, float(mi['model'].predict(fr)[0]))
        pt = pv * a90
        at = ar[target] * a90
        if pd.notna(at):
            errs.append(abs(pt - at))
            acts.append(at)
            preds.append(pt)

print(f"\n{'Metric':<20} {'Our Model':>12} {'Published':>12} {'Better?':>10}")
print("-" * 57)
gm = np.mean(ge)
am_val = np.mean(ae)
gr = r2_score(ag, pg)
ar_val = r2_score(aa, pa)
print(f"{'Goals MAE':<20} {gm:>12.2f} {'2.8':>12} {'✓' if gm < 2.8 else '✗':>10}")
print(f"{'Assists MAE':<20} {am_val:>12.2f} {'1.9':>12} {'✓' if am_val < 1.9 else '✗':>10}")
print(f"{'Goals R2':<20} {gr:>12.3f} {'~0.45-0.55':>12}")
print(f"{'Assists R2':<20} {ar_val:>12.3f} {'~0.35-0.45':>12}")


SECTION 1 — LOADING DATA
Outfield: 3868 rows
GK: 252 rows
Injuries: 15534 records
Players: 6051 records

SECTION 2 — CLEANING DATA
Outfield rows: 2777
GK rows: 180

Data quality check:
  2018: ✓ max=38 x90s
  2019: ✓ max=38 x90s
  2020: ✓ max=38 x90s
  2021: ✓ max=38 x90s
  2022: ✓ max=38 x90s
  2023: ✓ max=38 x90s
  2024: ✓ max=38 x90s

SECTION 3 — INJURY + MARKET VALUE FEATURES
Injury coverage: 2777/2777
Market value coverage: 2474/2777

SECTION 4 — LEAK-FREE SUB-POSITION CLUSTERING
Sub-position counts:
subposition
FB    586
CB    549
W     400
DM    357
ST    271
CM    244
AM    203
GK    167
Name: count, dtype: int64
Features: 25, Targets and params defined!
All helper functions defined!

SECTION 7 — BUILDING DATASETS
FW: 301 rows
DF: 580 rows
MF: 384 rows

SECTION 8 — TRAINING MODELS

--- FW ---
  goals_per_90: trained ✓  (train MAE: 0.091)
  assists_per_90: trained ✓  (train MAE: 0.064)
  xg_per_90: trained ✓  (train MAE: 0.065)
  shots_per_90: trained ✓  (train MAE: 0.320)
  nin

In [ ]:
# Fix age in 2024 data
print("Fixing age column in 2024 data...")
data_clean.loc[data_clean['season'] == 2024, 'age'] = \
    2024 - data_clean.loc[data_clean['season'] == 2024, 'born']

print(f"Age coverage after fix:")
age_2024 = data_clean[data_clean['season'] == 2024]['age']
print(f"  Non-NaN: {age_2024.notna().sum()}")
print(f"  NaN: {age_2024.isna().sum()}")
print("Done!")

Fixing age column in 2024 data...
Age coverage after fix:
  Non-NaN: 366
  NaN: 0
Done!


In [ ]:
print("=" * 55)
print("FIX 8 — AGING CURVE + DC TEAM STRENGTH")
print("=" * 55)

# Position-specific peak ages based on research
peak_ages_by_subpos = {
    'ST': 27, 'W': 26, 'AM': 28, 'CM': 28,
    'DM': 29, 'CB': 29, 'FB': 27, 'GK': 30
}

# Default peak by position group
peak_ages_by_pos = {'FW': 27, 'MF': 28, 'DF': 28, 'GK': 30}

def add_aging_features(data_df):
    data_df = data_df.copy()

    # Get peak age for each player based on subposition
    def get_peak(row):
        subpos = row.get('subposition', row.get('position_group', 'FW'))
        return peak_ages_by_subpos.get(subpos, peak_ages_by_pos.get(
            row.get('position_group', 'FW'), 27))

    data_df['peak_age'] = data_df.apply(get_peak, axis=1)
    data_df['age_vs_peak'] = data_df['age'] - data_df['peak_age']

    # Non-linear decline factor after peak
    def decline_factor(age_vs_peak):
        if pd.isna(age_vs_peak):
            return 1.0
        if age_vs_peak <= 0:
            return 1.0  # developing or at peak — no penalty
        elif age_vs_peak <= 2:
            return 0.97  # slight decline
        elif age_vs_peak <= 4:
            return 0.93  # moderate decline
        elif age_vs_peak <= 6:
            return 0.87  # significant decline
        else:
            return 0.80  # heavy decline

    data_df['decline_factor'] = data_df['age_vs_peak'].apply(decline_factor)

    # Binary flags
    data_df['is_developing'] = (data_df['age_vs_peak'] < -2).astype(int)
    data_df['is_peak_age'] = (data_df['age_vs_peak'].between(-2, 2)).astype(int)
    data_df['is_declining'] = (data_df['age_vs_peak'] > 2).astype(int)

    return data_df

data_clean = add_aging_features(data_clean)

print("Aging features added!")
print(f"\nSample aging features:")
sample_cols = ['player', 'season', 'age', 'peak_age', 'age_vs_peak',
               'decline_factor', 'is_developing', 'is_peak_age', 'is_declining']
sample_players = ['Alexander Isak', 'Mohamed Salah',
                  'Virgil van Dijk', 'Phil Foden']
for p in sample_players:
    row = data_clean[(data_clean['player'] == p) &
                     (data_clean['season'] == 2024)]
    if len(row) > 0:
        print(row[sample_cols].to_string())

# Add DC team strength from team_reports
print("\nAdding DC team strength features...")
team_reports = pd.read_csv(
    '/content/drive/MyDrive/player_ratings/data/team_reports.csv')

# Normalize team names for matching
team_reports['Team_clean'] = team_reports['Team'].str.strip()

# Create lookup dict
dc_attack_lookup = dict(zip(team_reports['Team_clean'],
                            team_reports['DC_Attack']))
dc_defence_lookup = dict(zip(team_reports['Team_clean'],
                             team_reports['DC_Defence']))

# Map team names between datasets
team_name_mapping = {
    'Manchester City': 'Manchester City',
    'Liverpool': 'Liverpool',
    'Arsenal': 'Arsenal',
    'Chelsea': 'Chelsea',
    'Aston Villa': 'Aston Villa',
    'Tottenham': 'Tottenham',
    'Newcastle Utd': 'Newcastle',
    'Manchester Utd': 'Manchester Utd',
    'West Ham': 'West Ham',
    'Brighton': 'Brighton',
    'Wolves': 'Wolves',
    'Fulham': 'Fulham',
    'Bournemouth': 'Bournemouth',
    'Crystal Palace': 'Crystal Palace',
    'Brentford': 'Brentford',
    'Everton': 'Everton',
    'Leicester City': 'Leicester City',
    'Southampton': 'Southampton',
    'Ipswich Town': 'Ipswich',
    "Nott'ham Forest": "Nott'm Forest",
}

# Add DC features to 2024 data only
data_clean['dc_attack'] = np.nan
data_clean['dc_defence'] = np.nan

for idx, row in data_clean[data_clean['season'] == 2024].iterrows():
    squad = row['squad']
    mapped = team_name_mapping.get(squad, squad)
    data_clean.at[idx, 'dc_attack'] = dc_attack_lookup.get(mapped, np.nan)
    data_clean.at[idx, 'dc_defence'] = dc_defence_lookup.get(mapped, np.nan)

coverage = data_clean[data_clean['season'] == 2024]['dc_attack'].notna().sum()
print(f"DC coverage for 2024: {coverage}/366 players")

# Update base_features to include new features
base_features_v3 = base_features + [
    'age_vs_peak', 'decline_factor',
    'is_developing', 'is_peak_age', 'is_declining',
    'dc_attack', 'dc_defence'
]

print(f"\nUpdated features: {len(base_features)} → {len(base_features_v3)}")
print("\nAging curve sample:")
print(data_clean[data_clean['season'] == 2024][
    ['player', 'age', 'age_vs_peak', 'decline_factor',
     'is_developing', 'is_declining']].head(10).to_string())

FIX 8 — AGING CURVE + DC TEAM STRENGTH
Aging features added!

Sample aging features:
              player  season   age  peak_age  age_vs_peak  decline_factor  is_developing  is_peak_age  is_declining
2425  Alexander Isak    2024  25.0        27         -2.0             1.0              0            1             0
             player  season   age  peak_age  age_vs_peak  decline_factor  is_developing  is_peak_age  is_declining
2663  Mohamed Salah    2024  32.0        27          5.0            0.87              0            0             1
               player  season   age  peak_age  age_vs_peak  decline_factor  is_developing  is_peak_age  is_declining
2757  Virgil van Dijk    2024  33.0        29          4.0            0.93              0            0             1
          player  season   age  peak_age  age_vs_peak  decline_factor  is_developing  is_peak_age  is_declining
2703  Phil Foden    2024  24.0        28         -4.0             1.0              1            0          

In [ ]:
subpos_to_pos = {'ST': 'FW', 'W': 'FW', 'CB': 'DF', 'FB': 'DF',
                 'DM': 'MF', 'CM': 'MF', 'AM': 'MF'}

subpos_targets_final = {
    'ST': ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'W':  ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'CB': ['aerials_won_pct', 'pass_completion_pct',
           'interceptions_per90', 'clearances_per90'],
    'FB': ['pass_completion_pct', 'interceptions_per90',
           'clearances_per90', 'assists_per_90'],
    'DM': ['tackles_interceptions_per90', 'ball_recoveries_per90',
           'pass_completion_pct'],
    'CM': ['progressive_passes_per90', 'tackles_interceptions_per90',
           'key_passes_per90', 'pass_completion_pct'],
    'AM': ['assists_per_90', 'key_passes_per90',
           'progressive_passes_per90', 'goals_per_90'],
}

pct_stats_final = ['aerials_won_pct', 'pass_completion_pct']

cv_mae_per_metric = {
    'goals_per_90': 2.32, 'assists_per_90': 1.64, 'xg_per_90': 1.69,
    'shots_per_90': 7.44, 'aerials_won_pct': 8.07, 'pass_completion_pct': 3.29,
    'interceptions_per90': 5.04, 'clearances_per90': 10.91,
    'progressive_passes_per90': 18.49, 'key_passes_per90': 6.88,
    'tackles_interceptions_per90': 11.40, 'ball_recoveries_per90': 16.73,
}

def get_confidence_interval(target, pred_val, pred_90s):
    mae = cv_mae_per_metric.get(target, 2.0)
    margin = round(mae * 1.5, 1)
    if target in pct_stats_final:
        return round(max(0, pred_val - margin), 1), round(pred_val + margin, 1)
    else:
        total = pred_val * pred_90s
        return round(max(0, total - margin), 1), round(total + margin, 1)

print("Variables defined!")

Variables defined!


In [ ]:
print("Retraining with aging curve + DC features...")

fw_full_v3 = build_dataset(data_clean, 'FW', up_to_season=2023)
df_full_v3 = build_dataset(data_clean, 'DF', up_to_season=2023)
mf_full_v3 = build_dataset(data_clean, 'MF', up_to_season=2023)

final_models_v3 = {}
feature_cols_v3 = [col for col in fw_full_v3.columns
                   if col not in ['player', 'season', 'position_group']
                   and not col.startswith('target_')]

for position, dataset in [('FW', fw_full_v3),
                           ('DF', df_full_v3),
                           ('MF', mf_full_v3)]:
    final_models_v3[position] = {}
    print(f"\n--- {position} ---")
    for target in position_targets_fixed[position] + ['ninety_mins']:
        target_col = f'target_{target}'
        if target_col not in dataset.columns:
            continue
        valid_cols = [col for col in feature_cols_v3 if col in dataset.columns]
        subset = dataset[valid_cols + [target_col]].dropna(subset=[target_col])
        if len(subset) < 10:
            continue
        X = subset[valid_cols].fillna(0)
        y = subset[target_col]
        model = XGBRegressor(**best_params_v2)
        model.fit(X, y)
        mae = mean_absolute_error(y, model.predict(X))
        final_models_v3[position][target] = {
            'model': model, 'feature_cols': valid_cols}
        print(f"  {target}: trained ✓  (MAE: {mae:.3f})")

# Test on 2024
print("\n" + "=" * 55)
print("TESTING V3 MODEL ON 2024")
print("=" * 55)

actual_2024 = data_clean[data_clean['season'] == 2024].copy()
fw_2024 = actual_2024[actual_2024['position_group'] == 'FW']
fw_hist = data_clean[(data_clean['position_group'] == 'FW') &
                     (data_clean['season'] < 2024)]

ge, ae, ag, aa, pg, pa = [], [], [], [], [], []

for player in fw_2024['player'].unique():
    features = create_multi_season_features(fw_hist, player, 2024)
    if features is None: continue
    ar = fw_2024[fw_2024['player'] == player]
    if len(ar) == 0: continue
    ar = ar.iloc[0]
    if pd.isna(ar.get('age', np.nan)): continue
    a90 = ar['ninety_mins_played']
    for target, errs, acts, preds in [
        ('goals_per_90', ge, ag, pg),
        ('assists_per_90', ae, aa, pa)]:
        if target not in final_models_v3['FW']: continue
        mi = final_models_v3['FW'][target]
        fr = pd.DataFrame([features])
        for col in mi['feature_cols']:
            if col not in fr.columns: fr[col] = 0
        fr = fr[mi['feature_cols']].fillna(0)
        pv = max(0, float(mi['model'].predict(fr)[0]))
        pt = pv * a90
        at = ar[target] * a90
        if pd.notna(at):
            errs.append(abs(pt - at))
            acts.append(at)
            preds.append(pt)

from sklearn.metrics import r2_score
gm = np.mean(ge)
am_val = np.mean(ae)
gr = r2_score(ag, pg)
ar_val = r2_score(aa, pa)

print(f"\n{'Metric':<20} {'V4':>10} {'V5 (aging)':>12} {'Published':>12} {'Better?':>10}")
print("-" * 67)
print(f"{'Goals MAE':<20} {'2.49':>10} {gm:>12.2f} {'2.8':>12} {'✓' if gm < 2.8 else '✗':>10}")
print(f"{'Assists MAE':<20} {'1.49':>10} {am_val:>12.2f} {'1.9':>12} {'✓' if am_val < 1.9 else '✗':>10}")
print(f"{'Goals R2':<20} {'0.751':>10} {gr:>12.3f} {'~0.45-0.55':>12}")
print(f"{'Assists R2':<20} {'0.552':>10} {ar_val:>12.3f} {'~0.35-0.45':>12}")

# Update final_models_v2 to use v3
final_models_v2 = final_models_v3
print("\nModels updated to v3!")

Retraining with aging curve + DC features...

--- FW ---
  goals_per_90: trained ✓  (MAE: 0.091)
  assists_per_90: trained ✓  (MAE: 0.064)
  xg_per_90: trained ✓  (MAE: 0.065)
  shots_per_90: trained ✓  (MAE: 0.320)
  ninety_mins: trained ✓  (MAE: 4.628)

--- DF ---
  aerials_won_pct: trained ✓  (MAE: 5.684)
  pass_completion_pct: trained ✓  (MAE: 2.422)
  interceptions_per90: trained ✓  (MAE: 0.227)
  clearances_per90: trained ✓  (MAE: 0.472)
  ninety_mins: trained ✓  (MAE: 5.526)

--- MF ---
  pass_completion_pct: trained ✓  (MAE: 2.075)
  progressive_passes_per90: trained ✓  (MAE: 0.710)
  key_passes_per90: trained ✓  (MAE: 0.259)
  tackles_interceptions_per90: trained ✓  (MAE: 0.418)
  ball_recoveries_per90: trained ✓  (MAE: 0.608)
  assists_per_90: trained ✓  (MAE: 0.052)
  ninety_mins: trained ✓  (MAE: 4.779)

TESTING V3 MODEL ON 2024

Metric                       V4   V5 (aging)    Published    Better?
-------------------------------------------------------------------
Goals MAE

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

def get_trend_explanation(df, player, subposition):
    ph = df[(df['player'] == player) & (df['season'] <= 2024)].tail(3)
    if len(ph) < 2:
        return "Limited data available for trend analysis."
    age = ph['age'].values[-1]
    if pd.isna(age):
        return "Limited data available for trend analysis."
    age = float(age)
    exp = []
    if age <= 23: exp.append(f"At {int(age)}, still in development years with room to grow.")
    elif age <= 27: exp.append(f"At {int(age)}, entering peak years.")
    elif age <= 30: exp.append(f"At {int(age)}, in prime years but may begin to slow.")
    else: exp.append(f"At {int(age)}, experience is key but decline is possible.")
    if subposition in ['ST', 'W'] and 'goals_per_90' in ph.columns:
        v = ph['goals_per_90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.05: exp.append("Goal rate improving season on season. ↑")
            elif t < -0.05: exp.append("Goal rate has declined recently. ↓")
            else: exp.append("Goal rate has been consistent. →")
    elif subposition in ['CM', 'AM'] and 'progressive_passes_per90' in ph.columns:
        v = ph['progressive_passes_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.5: exp.append("Progressive passing improving. ↑")
            elif t < -0.5: exp.append("Progressive passing declining. ↓")
            else: exp.append("Consistent in progressing play. →")
    elif subposition == 'DM' and 'tackles_interceptions_per90' in ph.columns:
        v = ph['tackles_interceptions_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive contribution improving. ↑")
            elif t < -0.3: exp.append("Defensive contribution declining. ↓")
            else: exp.append("Defensively consistent. →")
    elif subposition in ['CB', 'FB'] and 'clearances_per90' in ph.columns:
        v = ph['clearances_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive output improving. ↑")
            elif t < -0.3: exp.append("Defensive output declining. ↓")
            else: exp.append("Defensively consistent. →")
    mv = ph['ninety_mins_played'].dropna().values
    if len(mv) >= 2:
        if mv[-1] - mv[-2] > 5: exp.append("Playing more minutes each season.")
        elif mv[-1] - mv[-2] < -5: exp.append("Minutes dropped recently — injury or form concern.")
    return " ".join(exp)

# Generate predictions
print("Generating 2025 predictions...")
all_predictions_final = []

for subpos, targets in subpos_targets_final.items():
    position = subpos_to_pos[subpos]
    if position not in final_models_v2:
        continue
    subpos_players = data_clean[(data_clean['subposition'] == subpos) &
                                (data_clean['season'] == 2024)]['player'].unique()
    pos_hist = data_clean[(data_clean['position_group'] == position) &
                          (data_clean['season'] < 2024)]
    for player in subpos_players:
        features = create_multi_season_features(pos_hist, player, 2025)
        if features is None:
            continue
        recent = data_clean[(data_clean['player'] == player) &
                            (data_clean['season'] == 2024)]
        if len(recent) == 0:
            continue
        recent = recent.iloc[0]
        if pd.isna(recent.get('age', np.nan)):
            continue
        if 'ninety_mins' not in final_models_v2[position]:
            pred_90s = recent['ninety_mins_played']
        else:
            mi = final_models_v2[position]['ninety_mins']
            pred_90s = predict_minutes_v3(player, position, features,
                                          mi, data_clean)
        player_row = {
            'player': player,
            'squad': recent['squad'],
            'age': int(float(recent['age'])),
            'position': position,
            'subposition': subpos,
            'predicted_minutes': int(pred_90s * 90),
        }
        for target in targets:
            if target not in final_models_v2[position]:
                continue
            mi = final_models_v2[position][target]
            fr = pd.DataFrame([features])
            for col in mi['feature_cols']:
                if col not in fr.columns:
                    fr[col] = 0
            fr = fr[mi['feature_cols']].fillna(0)
            pv = max(0, float(mi['model'].predict(fr)[0]))
            if target in pct_stats_final:
                player_row[target] = round(pv, 1)
            else:
                player_row[target] = round(pv * pred_90s, 1)
            low, high = get_confidence_interval(target, pv, pred_90s)
            player_row[f'{target}_low'] = low
            player_row[f'{target}_high'] = high
        player_row['explanation'] = get_trend_explanation(
            data_clean, player, subpos)
        all_predictions_final.append(player_row)

predictions_final = pd.DataFrame(all_predictions_final)
print(f"Outfield predictions: {len(predictions_final)}")

# Add missing players including GKs
players_2024_all = data_clean[data_clean['season'] == 2024]['player'].unique()
missing = [p for p in players_2024_all
           if p not in predictions_final['player'].values]

extra = []
for player in missing:
    row = data_clean[(data_clean['player'] == player) &
                     (data_clean['season'] == 2024)].iloc[0]
    position = row['position_group']
    subpos = row['subposition']
    actual_90s = row['ninety_mins_played']
    if position == 'GK':
        gk_row = gk_data[(gk_data['player'] == player) &
                         (gk_data['season'] == 2024)]
        if len(gk_row) == 0:
            continue
        gk_row = gk_row.iloc[0]
        age = int(float(gk_row['age'])) if pd.notna(gk_row['age']) else 0
        if age <= 25: gk_exp = f"At {age}, young GK with development potential."
        elif age <= 30: gk_exp = f"At {age}, in peak years for a goalkeeper."
        else: gk_exp = f"At {age}, experienced GK relying on reading of the game."
        extra.append({
            'player': player, 'squad': gk_row['squad'], 'age': age,
            'position': 'GK', 'subposition': 'GK',
            'predicted_minutes': int(float(gk_row['ninety_mins_played']) * 90),
            'save_pct': round(float(gk_row.get('save_pct') or 0), 1),
            'clean_sheet_pct': round(float(gk_row.get('clean_sheet_pct') or 0), 1),
            'ga_per_90': round(float(gk_row.get('ga_per_90') or 0), 1),
            'explanation': gk_exp
        })
    else:
        targets_list = subpos_targets_final.get(subpos, [])
        pr = {
            'player': player, 'squad': row['squad'],
            'age': int(float(row['age'])) if pd.notna(row['age']) else 0,
            'position': position, 'subposition': subpos,
            'predicted_minutes': int(actual_90s * 90),
            'explanation': 'Prediction based on 2024 performance baseline.'
        }
        for target in targets_list:
            val = row.get(target, np.nan)
            if pd.isna(val):
                continue
            if target in pct_stats_final:
                pr[target] = round(float(val), 1)
            else:
                pr[target] = round(float(val) * actual_90s, 1)
        extra.append(pr)

if extra:
    predictions_final = pd.concat([predictions_final, pd.DataFrame(extra)],
                                   ignore_index=True)

print(f"Total players: {len(predictions_final)}")
print(predictions_final['subposition'].value_counts())

# Save to Drive
output_path = '/content/drive/MyDrive/player_ratings/'
predictions_final.to_csv(f'{output_path}final_predictions_2025_v5.csv', index=False)
for subpos in predictions_final['subposition'].unique():
    sdf = predictions_final[predictions_final['subposition'] == subpos].copy()
    sdf = sdf.dropna(axis=1, how='all')
    sdf.to_csv(f'{output_path}{subpos}_predictions_2025_v5.csv', index=False)

print("\nAll files saved! ✓")

print("\nSample ST:")
st_cols = ['player', 'squad', 'age', 'predicted_minutes',
           'goals_per_90', 'goals_per_90_low', 'goals_per_90_high',
           'assists_per_90', 'xg_per_90']
print(predictions_final[predictions_final['subposition'] == 'ST'][
    st_cols].head(5).to_string())

print("\nMinutes check:")
print(f"Max: {predictions_final['predicted_minutes'].max()}")
print(f"Avg: {predictions_final['predicted_minutes'].mean():.0f}")
print("\nTop 5 by minutes:")
print(predictions_final.nlargest(5, 'predicted_minutes')[
    ['player', 'subposition', 'predicted_minutes']].to_string())

Generating 2025 predictions...
Outfield predictions: 213
Total players: 366
subposition
FB    82
CB    72
DM    66
W     64
ST    37
AM    29
CM    16
Name: count, dtype: int64

All files saved! ✓

Sample ST:
            player          squad  age  predicted_minutes  goals_per_90  goals_per_90_low  goals_per_90_high  assists_per_90  xg_per_90
0   Alexander Isak  Newcastle Utd   25               1899          10.9               7.4               14.4             3.0       10.8
1  Antoine Semenyo    Bournemouth   24               2349           7.5               4.0               11.0             4.0        7.4
2             Beto        Everton   26               1170           4.2               0.7                7.7             1.9        6.0
3      Bukayo Saka        Arsenal   23               2232          10.7               7.2               14.2             5.5        9.4
4       Cody Gakpo      Liverpool   25               1485           6.5               3.0               10.0   

In [ ]:
print("=" * 55)
print("FIX 7 — ADDING ELO RATINGS")
print("=" * 55)

# Load Elo data
elo_df = pd.read_csv('/content/drive/MyDrive/player_ratings/data/elo_final_per_season.csv')
print(f"Elo records: {len(elo_df)}")
print(f"Seasons: {elo_df['season'].unique()}")
print(f"Sample:\n{elo_df.head(5).to_string()}")

# Convert season format 18_19 → 2019
def convert_elo_season(s):
    try:
        end = int(str(s).split('_')[1])
        return 2000 + end
    except:
        return None

elo_df['season_year'] = elo_df['season'].apply(convert_elo_season)

# Normalize team names to match our data
elo_team_mapping = {
    'Man United': 'Manchester Utd',
    'Man City': 'Manchester City',
    'Spurs': 'Tottenham',
    'Newcastle': 'Newcastle Utd',
    'Wolves': 'Wolves',
    "Nott'm Forest": "Nott'ham Forest",
    'Leicester': 'Leicester City',
    'Ipswich': 'Ipswich Town',
    'West Brom': 'West Brom',
    'Sheffield Utd': 'Sheffield Utd',
    'Leeds': 'Leeds United',
}

elo_df['team_clean'] = elo_df['team'].replace(elo_team_mapping)

# Create lookup: (team, season) → elo
elo_lookup = {}
for _, row in elo_df.iterrows():
    elo_lookup[(row['team_clean'], row['season_year'])] = row['final_elo']

print(f"\nElo lookup entries: {len(elo_lookup)}")

# Add Elo features to data_clean
print("\nAdding Elo features to data...")
data_clean['team_elo'] = np.nan
data_clean['team_elo_prev'] = np.nan
data_clean['team_elo_change'] = np.nan
data_clean['is_transfer'] = 0

for idx, row in data_clean.iterrows():
    player = row['player']
    season = row['season']
    squad = row['squad']

    # Current season Elo
    current_elo = elo_lookup.get((squad, season), np.nan)
    data_clean.at[idx, 'team_elo'] = current_elo

    # Previous season team
    prev_row = data_clean[(data_clean['player'] == player) &
                          (data_clean['season'] == season - 1)]
    if len(prev_row) > 0:
        prev_squad = prev_row.iloc[0]['squad']
        prev_elo = elo_lookup.get((prev_squad, season - 1), np.nan)
        data_clean.at[idx, 'team_elo_prev'] = prev_elo
        if pd.notna(current_elo) and pd.notna(prev_elo):
            data_clean.at[idx, 'team_elo_change'] = current_elo - prev_elo
        if prev_squad != squad:
            data_clean.at[idx, 'is_transfer'] = 1

elo_coverage = data_clean['team_elo'].notna().sum()
transfer_count = data_clean['is_transfer'].sum()
print(f"Elo coverage: {elo_coverage}/{len(data_clean)}")
print(f"Transfer seasons detected: {transfer_count}")

print("\nSample Elo features:")
sample_cols = ['player', 'season', 'squad', 'team_elo',
               'team_elo_change', 'is_transfer']
sample_players = ['Mohamed Salah', 'Alexander Isak',
                  'Erling Haaland', 'Virgil van Dijk']
for p in sample_players:
    rows = data_clean[data_clean['player'] == p][sample_cols]
    if len(rows) > 0:
        print(f"\n{p}:")
        print(rows.to_string())

# Update base features
base_features_v4 = base_features_v3 + [
    'team_elo', 'team_elo_change', 'is_transfer'
]
print(f"\nFeatures updated: {len(base_features_v3)} → {len(base_features_v4)}")
base_features = base_features_v4
print("Elo features added! ✓")

FIX 7 — ADDING ELO RATINGS
Elo records: 140
Seasons: ['18_19' '19_20' '20_21' '21_22' '22_23' '23_24' '24_25']
Sample:
  season         team  final_elo
0  18_19      Arsenal    1610.64
1  18_19  Bournemouth    1476.96
2  18_19     Brighton    1433.92
3  18_19      Burnley    1484.55
4  18_19      Cardiff    1420.49

Elo lookup entries: 140

Adding Elo features to data...
Elo coverage: 1983/2777
Transfer seasons detected: 155

Sample Elo features:

Mohamed Salah:
             player  season      squad  team_elo  team_elo_change  is_transfer
1658  Mohamed Salah    2018  Liverpool       NaN              NaN            0
1659  Mohamed Salah    2019  Liverpool   1774.84              NaN            0
1660  Mohamed Salah    2020  Liverpool   1819.68            44.84            0
1661  Mohamed Salah    2021  Liverpool   1732.34           -87.34            0
1662  Mohamed Salah    2022  Liverpool   1803.82            71.48            0
1663  Mohamed Salah    2023  Liverpool   1734.31           

In [ ]:
print("=" * 55)
print("FINAL RETRAIN WITH ALL FEATURES (Elo + Aging)")
print("=" * 55)

# Rebuild datasets with all new features
fw_final = build_dataset(data_clean, 'FW', up_to_season=2023)
df_final = build_dataset(data_clean, 'DF', up_to_season=2023)
mf_final = build_dataset(data_clean, 'MF', up_to_season=2023)

print(f"FW: {len(fw_final)} rows")
print(f"DF: {len(df_final)} rows")
print(f"MF: {len(mf_final)} rows")

final_models_v2 = {}
feature_cols_final = [col for col in fw_final.columns
                      if col not in ['player', 'season', 'position_group']
                      and not col.startswith('target_')]

for position, dataset in [('FW', fw_final), ('DF', df_final), ('MF', mf_final)]:
    final_models_v2[position] = {}
    print(f"\n--- {position} ---")
    for target in position_targets_fixed[position] + ['ninety_mins']:
        target_col = f'target_{target}'
        if target_col not in dataset.columns:
            continue
        valid_cols = [col for col in feature_cols_final if col in dataset.columns]
        subset = dataset[valid_cols + [target_col]].dropna(subset=[target_col])
        if len(subset) < 10:
            continue
        X = subset[valid_cols].fillna(0)
        y = subset[target_col]
        model = XGBRegressor(**best_params_v2)
        model.fit(X, y)
        mae = mean_absolute_error(y, model.predict(X))
        final_models_v2[position][target] = {
            'model': model, 'feature_cols': valid_cols}
        print(f"  {target}: trained ✓  (MAE: {mae:.3f})")

# Test on 2024
print("\n" + "=" * 55)
print("FINAL ACCURACY TEST ON 2024")
print("=" * 55)

actual_2024 = data_clean[data_clean['season'] == 2024].copy()
fw_2024 = actual_2024[actual_2024['position_group'] == 'FW']
fw_hist = data_clean[(data_clean['position_group'] == 'FW') &
                     (data_clean['season'] < 2024)]

ge, ae, ag, aa, pg, pa = [], [], [], [], [], []

for player in fw_2024['player'].unique():
    features = create_multi_season_features(fw_hist, player, 2024)
    if features is None: continue
    ar = fw_2024[fw_2024['player'] == player]
    if len(ar) == 0: continue
    ar = ar.iloc[0]
    if pd.isna(ar.get('age', np.nan)): continue
    a90 = ar['ninety_mins_played']
    for target, errs, acts, preds in [
        ('goals_per_90', ge, ag, pg),
        ('assists_per_90', ae, aa, pa)]:
        if target not in final_models_v2['FW']: continue
        mi = final_models_v2['FW'][target]
        fr = pd.DataFrame([features])
        for col in mi['feature_cols']:
            if col not in fr.columns: fr[col] = 0
        fr = fr[mi['feature_cols']].fillna(0)
        pv = max(0, float(mi['model'].predict(fr)[0]))
        pt = pv * a90
        at = ar[target] * a90
        if pd.notna(at):
            errs.append(abs(pt - at))
            acts.append(at)
            preds.append(pt)

from sklearn.metrics import r2_score
gm = np.mean(ge)
am_val = np.mean(ae)
gr = r2_score(ag, pg)
ar_val = r2_score(aa, pa)

print(f"\n{'Metric':<20} {'V4':>8} {'V5':>8} {'Final':>8} {'Published':>12} {'Better?':>10}")
print("-" * 70)
print(f"{'Goals MAE':<20} {'2.49':>8} {'2.49':>8} {gm:>8.2f} {'2.8':>12} {'✓' if gm < 2.8 else '✗':>10}")
print(f"{'Assists MAE':<20} {'1.49':>8} {'1.49':>8} {am_val:>8.2f} {'1.9':>12} {'✓' if am_val < 1.9 else '✗':>10}")
print(f"{'Goals R2':<20} {'0.751':>8} {'0.751':>8} {gr:>8.3f} {'~0.45-0.55':>12}")
print(f"{'Assists R2':<20} {'0.552':>8} {'0.552':>8} {ar_val:>8.3f} {'~0.35-0.45':>12}")
print(f"\nPlayers evaluated: {len(ge)}")

FINAL RETRAIN WITH ALL FEATURES (Elo + Aging)
FW: 301 rows
DF: 580 rows
MF: 384 rows

--- FW ---
  goals_per_90: trained ✓  (MAE: 0.091)
  assists_per_90: trained ✓  (MAE: 0.064)
  xg_per_90: trained ✓  (MAE: 0.065)
  shots_per_90: trained ✓  (MAE: 0.319)
  ninety_mins: trained ✓  (MAE: 4.554)

--- DF ---
  aerials_won_pct: trained ✓  (MAE: 5.688)
  pass_completion_pct: trained ✓  (MAE: 2.420)
  interceptions_per90: trained ✓  (MAE: 0.226)
  clearances_per90: trained ✓  (MAE: 0.472)
  ninety_mins: trained ✓  (MAE: 5.516)

--- MF ---
  pass_completion_pct: trained ✓  (MAE: 2.089)
  progressive_passes_per90: trained ✓  (MAE: 0.709)
  key_passes_per90: trained ✓  (MAE: 0.259)
  tackles_interceptions_per90: trained ✓  (MAE: 0.418)
  ball_recoveries_per90: trained ✓  (MAE: 0.603)
  assists_per_90: trained ✓  (MAE: 0.052)
  ninety_mins: trained ✓  (MAE: 4.756)

FINAL ACCURACY TEST ON 2024

Metric                     V4       V5    Final    Published    Better?
-------------------------------

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

def get_trend_explanation(df, player, subposition):
    ph = df[(df['player'] == player) & (df['season'] <= 2024)].tail(3)
    if len(ph) < 2:
        return "Limited data available for trend analysis."
    age = ph['age'].values[-1]
    if pd.isna(age):
        return "Limited data available for trend analysis."
    age = float(age)
    exp = []
    if age <= 23: exp.append(f"At {int(age)}, still in development years with room to grow.")
    elif age <= 27: exp.append(f"At {int(age)}, entering peak years.")
    elif age <= 30: exp.append(f"At {int(age)}, in prime years but may begin to slow.")
    else: exp.append(f"At {int(age)}, experience is key but decline is possible.")
    if subposition in ['ST', 'W'] and 'goals_per_90' in ph.columns:
        v = ph['goals_per_90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.05: exp.append("Goal rate improving season on season. ↑")
            elif t < -0.05: exp.append("Goal rate has declined recently. ↓")
            else: exp.append("Goal rate has been consistent. →")
    elif subposition in ['CM', 'AM'] and 'progressive_passes_per90' in ph.columns:
        v = ph['progressive_passes_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.5: exp.append("Progressive passing improving. ↑")
            elif t < -0.5: exp.append("Progressive passing declining. ↓")
            else: exp.append("Consistent in progressing play. →")
    elif subposition == 'DM' and 'tackles_interceptions_per90' in ph.columns:
        v = ph['tackles_interceptions_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive contribution improving. ↑")
            elif t < -0.3: exp.append("Defensive contribution declining. ↓")
            else: exp.append("Defensively consistent. →")
    elif subposition in ['CB', 'FB'] and 'clearances_per90' in ph.columns:
        v = ph['clearances_per90'].dropna().values
        if len(v) >= 2:
            t = v[-1] - v[-2]
            if t > 0.3: exp.append("Defensive output improving. ↑")
            elif t < -0.3: exp.append("Defensive output declining. ↓")
            else: exp.append("Defensively consistent. →")
    mv = ph['ninety_mins_played'].dropna().values
    if len(mv) >= 2:
        if mv[-1] - mv[-2] > 5: exp.append("Playing more minutes each season.")
        elif mv[-1] - mv[-2] < -5: exp.append("Minutes dropped recently — injury or form concern.")
    return " ".join(exp)

subpos_to_pos = {'ST': 'FW', 'W': 'FW', 'CB': 'DF', 'FB': 'DF',
                 'DM': 'MF', 'CM': 'MF', 'AM': 'MF'}

subpos_targets_final = {
    'ST': ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'W':  ['goals_per_90', 'assists_per_90', 'xg_per_90', 'shots_per_90'],
    'CB': ['aerials_won_pct', 'pass_completion_pct',
           'interceptions_per90', 'clearances_per90'],
    'FB': ['pass_completion_pct', 'interceptions_per90',
           'clearances_per90', 'assists_per_90'],
    'DM': ['tackles_interceptions_per90', 'ball_recoveries_per90',
           'pass_completion_pct'],
    'CM': ['progressive_passes_per90', 'tackles_interceptions_per90',
           'key_passes_per90', 'pass_completion_pct'],
    'AM': ['assists_per_90', 'key_passes_per90',
           'progressive_passes_per90', 'goals_per_90'],
}

pct_stats_final = ['aerials_won_pct', 'pass_completion_pct']

cv_mae_per_metric = {
    'goals_per_90': 2.32, 'assists_per_90': 1.64, 'xg_per_90': 1.69,
    'shots_per_90': 7.44, 'aerials_won_pct': 8.07, 'pass_completion_pct': 3.29,
    'interceptions_per90': 5.04, 'clearances_per90': 10.91,
    'progressive_passes_per90': 18.49, 'key_passes_per90': 6.88,
    'tackles_interceptions_per90': 11.40, 'ball_recoveries_per90': 16.73,
}

def get_confidence_interval(target, pred_val, pred_90s):
    mae = cv_mae_per_metric.get(target, 2.0)
    margin = round(mae * 1.5, 1)
    if target in pct_stats_final:
        return round(max(0, pred_val - margin), 1), round(pred_val + margin, 1)
    else:
        total = pred_val * pred_90s
        return round(max(0, total - margin), 1), round(total + margin, 1)

# Generate predictions
print("Generating final 2025 predictions (v6)...")
all_predictions_final = []

for subpos, targets in subpos_targets_final.items():
    position = subpos_to_pos[subpos]
    if position not in final_models_v2:
        continue
    subpos_players = data_clean[(data_clean['subposition'] == subpos) &
                                (data_clean['season'] == 2024)]['player'].unique()
    pos_hist = data_clean[(data_clean['position_group'] == position) &
                          (data_clean['season'] < 2024)]
    for player in subpos_players:
        features = create_multi_season_features(pos_hist, player, 2025)
        if features is None:
            continue
        recent = data_clean[(data_clean['player'] == player) &
                            (data_clean['season'] == 2024)]
        if len(recent) == 0:
            continue
        recent = recent.iloc[0]
        if pd.isna(recent.get('age', np.nan)):
            continue
        if 'ninety_mins' not in final_models_v2[position]:
            pred_90s = recent['ninety_mins_played']
        else:
            mi = final_models_v2[position]['ninety_mins']
            pred_90s = predict_minutes_v3(player, position, features, mi, data_clean)
        player_row = {
            'player': player,
            'squad': recent['squad'],
            'age': int(float(recent['age'])),
            'position': position,
            'subposition': subpos,
            'predicted_minutes': int(pred_90s * 90),
        }
        for target in targets:
            if target not in final_models_v2[position]:
                continue
            mi = final_models_v2[position][target]
            fr = pd.DataFrame([features])
            for col in mi['feature_cols']:
                if col not in fr.columns:
                    fr[col] = 0
            fr = fr[mi['feature_cols']].fillna(0)
            pv = max(0, float(mi['model'].predict(fr)[0]))
            if target in pct_stats_final:
                player_row[target] = round(pv, 1)
            else:
                player_row[target] = round(pv * pred_90s, 1)
            low, high = get_confidence_interval(target, pv, pred_90s)
            player_row[f'{target}_low'] = low
            player_row[f'{target}_high'] = high
        player_row['explanation'] = get_trend_explanation(
            data_clean, player, subpos)
        all_predictions_final.append(player_row)

predictions_final = pd.DataFrame(all_predictions_final)
print(f"Outfield predictions: {len(predictions_final)}")

# Add missing players including GKs
players_2024_all = data_clean[data_clean['season'] == 2024]['player'].unique()
missing = [p for p in players_2024_all
           if p not in predictions_final['player'].values]

extra = []
for player in missing:
    row = data_clean[(data_clean['player'] == player) &
                     (data_clean['season'] == 2024)].iloc[0]
    position = row['position_group']
    subpos = row['subposition']
    actual_90s = row['ninety_mins_played']
    if position == 'GK':
        gk_row = gk_data[(gk_data['player'] == player) &
                         (gk_data['season'] == 2024)]
        if len(gk_row) == 0:
            continue
        gk_row = gk_row.iloc[0]
        age = int(float(gk_row['age'])) if pd.notna(gk_row['age']) else 0
        if age <= 25: gk_exp = f"At {age}, young GK with development potential."
        elif age <= 30: gk_exp = f"At {age}, in peak years for a goalkeeper."
        else: gk_exp = f"At {age}, experienced GK relying on reading of the game."
        extra.append({
            'player': player, 'squad': gk_row['squad'], 'age': age,
            'position': 'GK', 'subposition': 'GK',
            'predicted_minutes': int(float(gk_row['ninety_mins_played']) * 90),
            'save_pct': round(float(gk_row.get('save_pct') or 0), 1),
            'clean_sheet_pct': round(float(gk_row.get('clean_sheet_pct') or 0), 1),
            'ga_per_90': round(float(gk_row.get('ga_per_90') or 0), 1),
            'explanation': gk_exp
        })
    else:
        targets_list = subpos_targets_final.get(subpos, [])
        pr = {
            'player': player, 'squad': row['squad'],
            'age': int(float(row['age'])) if pd.notna(row['age']) else 0,
            'position': position, 'subposition': subpos,
            'predicted_minutes': int(actual_90s * 90),
            'explanation': 'Prediction based on 2024 performance baseline.'
        }
        for target in targets_list:
            val = row.get(target, np.nan)
            if pd.isna(val):
                continue
            if target in pct_stats_final:
                pr[target] = round(float(val), 1)
            else:
                pr[target] = round(float(val) * actual_90s, 1)
        extra.append(pr)

if extra:
    predictions_final = pd.concat([predictions_final, pd.DataFrame(extra)],
                                   ignore_index=True)

print(f"Total players: {len(predictions_final)}")
print(predictions_final['subposition'].value_counts())

# Save to Drive as v6
output_path = '/content/drive/MyDrive/player_ratings/'
predictions_final.to_csv(f'{output_path}final_predictions_2025_v6.csv', index=False)
for subpos in predictions_final['subposition'].unique():
    sdf = predictions_final[predictions_final['subposition'] == subpos].copy()
    sdf = sdf.dropna(axis=1, how='all')
    sdf.to_csv(f'{output_path}{subpos}_predictions_2025_v6.csv', index=False)

print("\nAll files saved as v6! ✓")

print("\nSample ST:")
st_cols = ['player', 'squad', 'age', 'predicted_minutes',
           'goals_per_90', 'goals_per_90_low', 'goals_per_90_high',
           'assists_per_90', 'xg_per_90']
print(predictions_final[predictions_final['subposition'] == 'ST'][
    st_cols].head(5).to_string())

print("\nMinutes check:")
print(f"Max: {predictions_final['predicted_minutes'].max()}")
print(f"Avg: {predictions_final['predicted_minutes'].mean():.0f}")
print("\nTop 5 by minutes:")
print(predictions_final.nlargest(5, 'predicted_minutes')[
    ['player', 'subposition', 'predicted_minutes']].to_string())

Generating final 2025 predictions (v6)...
Outfield predictions: 213
Total players: 366
subposition
FB    82
CB    72
DM    66
W     64
ST    37
AM    29
CM    16
Name: count, dtype: int64

All files saved as v6! ✓

Sample ST:
            player          squad  age  predicted_minutes  goals_per_90  goals_per_90_low  goals_per_90_high  assists_per_90  xg_per_90
0   Alexander Isak  Newcastle Utd   25               1890          10.8               7.3               14.3             3.1       11.0
1  Antoine Semenyo    Bournemouth   24               2367           7.6               4.1               11.1             4.0        7.2
2             Beto        Everton   26               1179           4.2               0.7                7.7             1.9        6.0
3      Bukayo Saka        Arsenal   23               2241          11.2               7.7               14.7             6.0        9.7
4       Cody Gakpo      Liverpool   25               1475           6.5               3.0     

In [ ]:
print("=" * 55)
print("GK ML MODEL")
print("=" * 55)

# Clean GK data
gk_data_clean = gk_data[gk_data['ninety_mins_played'] >= 5].copy()
gk_data_clean = gk_data_clean.sort_values(['player', 'season']).reset_index(drop=True)

# Add aging features to GK data
gk_data_clean['age'] = gk_data_clean['age'].fillna(
    2024 - gk_data_clean['born'])
gk_data_clean['peak_age'] = 30  # GKs peak at 30
gk_data_clean['age_vs_peak'] = gk_data_clean['age'] - 30
gk_data_clean['is_developing'] = (gk_data_clean['age_vs_peak'] < -3).astype(int)
gk_data_clean['is_declining'] = (gk_data_clean['age_vs_peak'] > 3).astype(int)

# Add Elo to GK data
gk_data_clean['team_elo'] = np.nan
gk_data_clean['team_elo_change'] = np.nan

for idx, row in gk_data_clean.iterrows():
    squad = row['squad']
    season = row['season']
    mapped = team_name_mapping.get(squad, squad)
    current_elo = elo_lookup.get((mapped, season), np.nan)
    gk_data_clean.at[idx, 'team_elo'] = current_elo
    prev = gk_data_clean[(gk_data_clean['player'] == row['player']) &
                          (gk_data_clean['season'] == season - 1)]
    if len(prev) > 0:
        prev_squad = prev.iloc[0]['squad']
        prev_mapped = team_name_mapping.get(prev_squad, prev_squad)
        prev_elo = elo_lookup.get((prev_mapped, season - 1), np.nan)
        if pd.notna(current_elo) and pd.notna(prev_elo):
            gk_data_clean.at[idx, 'team_elo_change'] = current_elo - prev_elo

print(f"GK rows: {len(gk_data_clean)}")
print(f"Seasons: {sorted(gk_data_clean['season'].unique())}")

# GK features
gk_features = ['age', 'ninety_mins_played', 'save_pct', 'clean_sheet_pct',
               'ga_per_90', 'post_shot_xg_minus_ga_per_90',
               'crosses_stopped_pct', 'defensive_opa_per_90',
               'age_vs_peak', 'is_developing', 'is_declining',
               'team_elo', 'team_elo_change']

# GK targets
gk_targets = ['save_pct', 'clean_sheet_pct',
              'ga_per_90', 'post_shot_xg_minus_ga_per_90']

# Create multi-season features for GKs
def create_gk_features(df, player, current_season):
    all_data = df[(df['player'] == player) &
                  (df['season'] < current_season)].sort_values('season')
    if len(all_data) == 0:
        return None
    recent = all_data.tail(3)
    features = {
        'seasons_available': len(recent),
        'total_gk_seasons': len(all_data)
    }
    for feat in gk_features:
        if feat not in all_data.columns:
            continue
        av = all_data[feat].values.astype(float)
        rv = recent[feat].values.astype(float)
        features[f'{feat}_s1'] = rv[-1] if len(rv) >= 1 else np.nan
        features[f'{feat}_s2'] = rv[-2] if len(rv) >= 2 else np.nan
        features[f'{feat}_s3'] = rv[-3] if len(rv) >= 3 else np.nan
        features[f'{feat}_trend'] = (rv[-1] - rv[-2]) if len(rv) >= 2 else np.nan
        features[f'{feat}_avg'] = np.nanmean(av)
        vv = av[~np.isnan(av)]
        features[f'{feat}_career_best'] = np.nanmax(av) if len(vv) > 0 else np.nan
    features['age'] = all_data['age'].values[-1]
    features['last_season_minutes'] = all_data['ninety_mins_played'].values[-1]
    return features

# Build GK dataset
print("\nBuilding GK dataset...")
gk_rows = []
gk_pre2024 = gk_data_clean[gk_data_clean['season'] < 2024].copy()

for player in gk_pre2024['player'].unique():
    for season in sorted(gk_pre2024[gk_pre2024['player'] == player]['season'].unique()):
        features = create_gk_features(gk_pre2024, player, season)
        if features is None:
            continue
        actual = gk_pre2024[(gk_pre2024['player'] == player) &
                            (gk_pre2024['season'] == season)]
        if len(actual) == 0:
            continue
        actual = actual.iloc[0]
        for target in gk_targets + ['ninety_mins_played']:
            if target in actual:
                features[f'target_{target}'] = actual[target]
        features['player'] = player
        features['season'] = season
        gk_rows.append(features)

gk_dataset = pd.DataFrame(gk_rows)
print(f"GK dataset: {len(gk_dataset)} rows")

# Train GK models
gk_models = {}
gk_feature_cols = [col for col in gk_dataset.columns
                   if col not in ['player', 'season']
                   and not col.startswith('target_')]

print("\nTraining GK models...")
for target in gk_targets + ['ninety_mins_played']:
    target_col = f'target_{target}'
    if target_col not in gk_dataset.columns:
        continue
    valid_cols = [col for col in gk_feature_cols if col in gk_dataset.columns]
    subset = gk_dataset[valid_cols + [target_col]].dropna(subset=[target_col])
    if len(subset) < 10:
        print(f"  {target}: not enough data")
        continue
    X = subset[valid_cols].fillna(0)
    y = subset[target_col]
    model = XGBRegressor(**best_params_v2)
    model.fit(X, y)
    mae = mean_absolute_error(y, model.predict(X))
    gk_models[target] = {'model': model, 'feature_cols': valid_cols}
    print(f"  {target}: trained ✓  (MAE: {mae:.3f})")

# Generate GK predictions for 2025
print("\nGenerating GK 2025 predictions...")
gk_predictions = []
gk_2024 = gk_data_clean[gk_data_clean['season'] == 2024].copy()

for player in gk_2024['player'].unique():
    features = create_gk_features(gk_data_clean, player, 2025)

    recent = gk_2024[gk_2024['player'] == player]
    if len(recent) == 0:
        continue
    recent = recent.iloc[0]

    age = int(float(recent['age'])) if pd.notna(recent['age']) else 0

    if age <= 25: gk_exp = f"At {age}, young GK with development potential."
    elif age <= 30: gk_exp = f"At {age}, in peak years for a goalkeeper."
    else: gk_exp = f"At {age}, experienced GK — reading of game is key."

    if features is None:
        # No history — use 2024 stats as baseline
        gk_predictions.append({
            'player': player,
            'squad': recent['squad'],
            'age': age,
            'position': 'GK',
            'subposition': 'GK',
            'predicted_minutes': int(float(recent['ninety_mins_played']) * 90),
            'save_pct': round(float(recent.get('save_pct') or 0), 1),
            'clean_sheet_pct': round(float(recent.get('clean_sheet_pct') or 0), 1),
            'ga_per_90': round(float(recent.get('ga_per_90') or 0), 1),
            'post_shot_xg_minus_ga_per_90': round(
                float(recent.get('post_shot_xg_minus_ga_per_90') or 0), 2),
            'explanation': gk_exp + ' Limited data — based on 2024 performance.'
        })
        continue

    # Predict minutes
    pred_90s = recent['ninety_mins_played']
    if 'ninety_mins_played' in gk_models:
        mi = gk_models['ninety_mins_played']
        fr = pd.DataFrame([features])
        for col in mi['feature_cols']:
            if col not in fr.columns:
                fr[col] = 0
        fr = fr[mi['feature_cols']].fillna(0)
        pred_90s = max(5, min(38, float(mi['model'].predict(fr)[0])))

    player_row = {
        'player': player,
        'squad': recent['squad'],
        'age': age,
        'position': 'GK',
        'subposition': 'GK',
        'predicted_minutes': int(pred_90s * 90),
        'explanation': gk_exp
    }

    for target in gk_targets:
        if target not in gk_models:
            continue
        mi = gk_models[target]
        fr = pd.DataFrame([features])
        for col in mi['feature_cols']:
            if col not in fr.columns:
                fr[col] = 0
        fr = fr[mi['feature_cols']].fillna(0)
        pv = float(mi['model'].predict(fr)[0])
        player_row[target] = round(pv, 3)

    gk_predictions.append(player_row)

gk_pred_df = pd.DataFrame(gk_predictions)
print(f"GK predictions: {len(gk_pred_df)}")
print(gk_pred_df[['player', 'squad', 'age', 'predicted_minutes',
                   'save_pct', 'clean_sheet_pct', 'ga_per_90']].to_string())

# Save GK predictions
output_path = '/content/drive/MyDrive/player_ratings/'
gk_pred_df.to_csv(f'{output_path}GK_predictions_2025_v6.csv', index=False)
print("\nGK predictions saved! ✓")

GK ML MODEL
GK rows: 180
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Building GK dataset...
GK dataset: 83 rows

Training GK models...
  save_pct: trained ✓  (MAE: 2.063)
  clean_sheet_pct: trained ✓  (MAE: 4.149)
  ga_per_90: trained ✓  (MAE: 0.156)
  post_shot_xg_minus_ga_per_90: trained ✓  (MAE: 0.081)
  ninety_mins_played: trained ✓  (MAE: 4.291)

Generating GK 2025 predictions...
GK predictions: 27
               player            squad  age  predicted_minutes  save_pct  clean_sheet_pct  ga_per_90
0      Aaron Ramsdale      Southampton   26               2155    69.754           29.174      1.356
1       Alex McCarthy      Southampton   35               1903    65.804           24.566      1.742
2             Alisson        Liverpool   32               2947    72.961           34.273      1.085
3     Alphonse Areola         West Ham   31               2539    68.742           25.127      1.557
4        

In [ ]:
# Update final predictions with proper GK ML predictions
print("Updating final predictions with GK ML model...")

# Remove old GK rows from predictions_final
predictions_no_gk = predictions_final[predictions_final['subposition'] != 'GK'].copy()

# Add new GK ML predictions
predictions_final_v6 = pd.concat([predictions_no_gk, gk_pred_df],
                                   ignore_index=True)

print(f"Total players: {len(predictions_final_v6)}")
print(predictions_final_v6['subposition'].value_counts())

# Save updated final file
output_path = '/content/drive/MyDrive/player_ratings/'
predictions_final_v6.to_csv(
    f'{output_path}final_predictions_2025_v6.csv', index=False)

# Save separate GK file
gk_pred_df.to_csv(f'{output_path}GK_predictions_2025_v6.csv', index=False)

print("\nFinal v6 saved with proper GK ML predictions! ✓")
print("\nSample GK comparison (old vs new):")
print("Old: raw 2024 stats")
print("New: ML predicted 2025 stats")
print(gk_pred_df[['player', 'squad', 'age', 'predicted_minutes',
                   'save_pct', 'clean_sheet_pct', 'ga_per_90']].head(5).to_string())

Updating final predictions with GK ML model...
Total players: 393
subposition
FB    82
CB    72
DM    66
W     64
ST    37
AM    29
GK    27
CM    16
Name: count, dtype: int64

Final v6 saved with proper GK ML predictions! ✓

Sample GK comparison (old vs new):
Old: raw 2024 stats
New: ML predicted 2025 stats
            player           squad  age  predicted_minutes  save_pct  clean_sheet_pct  ga_per_90
0   Aaron Ramsdale     Southampton   26               2155    69.754           29.174      1.356
1    Alex McCarthy     Southampton   35               1903    65.804           24.566      1.742
2          Alisson       Liverpool   32               2947    72.961           34.273      1.085
3  Alphonse Areola        West Ham   31               2539    68.742           25.127      1.557
4      Andre Onana  Manchester Utd   28               3029    69.186           30.716      1.329


In [ ]:
print("=" * 55)
print("BUILDING FINAL COMPARISON FILE")
print("=" * 55)

# Get actual 2024 stats for all players
actual_2024 = data_clean[data_clean['season'] == 2024].copy()
actual_2024_gk = gk_data_clean[gk_data_clean['season'] == 2024].copy()

comparison_rows = []

for _, pred_row in predictions_final_v6.iterrows():
    player = pred_row['player']
    subpos = pred_row['subposition']
    position = pred_row['position']

    row = {
        'player': player,
        'squad': pred_row.get('squad', ''),
        'age': pred_row.get('age', ''),
        'position': position,
        'subposition': subpos,
    }

    if position == 'GK':
        actual = actual_2024_gk[actual_2024_gk['player'] == player]
        if len(actual) > 0:
            actual = actual.iloc[0]
            a90s = actual['ninety_mins_played']
            # Actual 2024
            row['actual_2024_minutes'] = int(a90s * 90)
            row['actual_2024_save_pct'] = round(float(actual.get('save_pct') or 0), 1)
            row['actual_2024_clean_sheet_pct'] = round(float(actual.get('clean_sheet_pct') or 0), 1)
            row['actual_2024_ga_per_90'] = round(float(actual.get('ga_per_90') or 0), 2)
        # Predicted 2025
        row['predicted_2025_minutes'] = pred_row.get('predicted_minutes', '')
        row['predicted_2025_save_pct'] = pred_row.get('save_pct', '')
        row['predicted_2025_clean_sheet_pct'] = pred_row.get('clean_sheet_pct', '')
        row['predicted_2025_ga_per_90'] = pred_row.get('ga_per_90', '')
        row['explanation'] = pred_row.get('explanation', '')

    else:
        actual = actual_2024[actual_2024['player'] == player]
        if len(actual) > 0:
            actual = actual.iloc[0]
            a90s = actual['ninety_mins_played']

            if subpos in ['ST', 'W']:
                # Per90
                row['actual_2024_goals_per90'] = round(float(actual.get('goals_per_90') or 0), 2)
                row['actual_2024_assists_per90'] = round(float(actual.get('assists_per_90') or 0), 2)
                row['actual_2024_xg_per90'] = round(float(actual.get('xg_per_90') or 0), 2)
                row['actual_2024_shots_per90'] = round(float(actual.get('shots_per_90') or 0), 2)
                # Totals
                row['actual_2024_minutes'] = int(a90s * 90)
                row['actual_2024_goals'] = round(float(actual.get('goals_per_90') or 0) * a90s, 1)
                row['actual_2024_assists'] = round(float(actual.get('assists_per_90') or 0) * a90s, 1)
                row['actual_2024_xg'] = round(float(actual.get('xg_per_90') or 0) * a90s, 1)
                row['actual_2024_shots'] = round(float(actual.get('shots_per_90') or 0) * a90s, 1)

            elif subpos in ['CB', 'FB']:
                row['actual_2024_aerials_won_pct'] = round(float(actual.get('aerials_won_pct') or 0), 1)
                row['actual_2024_pass_completion_pct'] = round(float(actual.get('pass_completion_pct') or 0), 1)
                row['actual_2024_interceptions_per90'] = round(float(actual.get('interceptions_per90') or 0), 2)
                row['actual_2024_clearances_per90'] = round(float(actual.get('clearances_per90') or 0), 2)
                row['actual_2024_minutes'] = int(a90s * 90)
                row['actual_2024_interceptions'] = round(float(actual.get('interceptions_per90') or 0) * a90s, 1)
                row['actual_2024_clearances'] = round(float(actual.get('clearances_per90') or 0) * a90s, 1)

            elif subpos in ['DM', 'CM', 'AM']:
                row['actual_2024_assists_per90'] = round(float(actual.get('assists_per_90') or 0), 2)
                row['actual_2024_progressive_passes_per90'] = round(float(actual.get('progressive_passes_per90') or 0), 2)
                row['actual_2024_key_passes_per90'] = round(float(actual.get('key_passes_per90') or 0), 2)
                row['actual_2024_tackles_interceptions_per90'] = round(float(actual.get('tackles_interceptions_per90') or 0), 2)
                row['actual_2024_minutes'] = int(a90s * 90)
                row['actual_2024_assists'] = round(float(actual.get('assists_per_90') or 0) * a90s, 1)
                row['actual_2024_progressive_passes'] = round(float(actual.get('progressive_passes_per90') or 0) * a90s, 1)
                row['actual_2024_key_passes'] = round(float(actual.get('key_passes_per90') or 0) * a90s, 1)
                row['actual_2024_tackles_interceptions'] = round(float(actual.get('tackles_interceptions_per90') or 0) * a90s, 1)

        # Add predicted 2025 stats
        row['predicted_2025_minutes'] = pred_row.get('predicted_minutes', '')

        if subpos in ['ST', 'W']:
            row['predicted_2025_goals_per90'] = pred_row.get('goals_per_90', '')
            row['predicted_2025_assists_per90'] = pred_row.get('assists_per_90', '')
            row['predicted_2025_xg_per90'] = pred_row.get('xg_per_90', '')
            row['predicted_2025_shots_per90'] = pred_row.get('shots_per_90', '')
            row['predicted_2025_goals'] = pred_row.get('goals_per_90', '')
            row['predicted_2025_assists'] = pred_row.get('assists_per_90', '')
            row['predicted_2025_xg'] = pred_row.get('xg_per_90', '')
            row['predicted_2025_shots'] = pred_row.get('shots_per_90', '')
            row['predicted_2025_goals_low'] = pred_row.get('goals_per_90_low', '')
            row['predicted_2025_goals_high'] = pred_row.get('goals_per_90_high', '')

        elif subpos in ['CB', 'FB']:
            row['predicted_2025_aerials_won_pct'] = pred_row.get('aerials_won_pct', '')
            row['predicted_2025_pass_completion_pct'] = pred_row.get('pass_completion_pct', '')
            row['predicted_2025_interceptions_per90'] = pred_row.get('interceptions_per90', '')
            row['predicted_2025_clearances_per90'] = pred_row.get('clearances_per90', '')
            row['predicted_2025_interceptions'] = pred_row.get('interceptions_per90', '')
            row['predicted_2025_clearances'] = pred_row.get('clearances_per90', '')

        elif subpos in ['DM', 'CM', 'AM']:
            row['predicted_2025_assists_per90'] = pred_row.get('assists_per_90', '')
            row['predicted_2025_progressive_passes_per90'] = pred_row.get('progressive_passes_per90', '')
            row['predicted_2025_key_passes_per90'] = pred_row.get('key_passes_per90', '')
            row['predicted_2025_tackles_interceptions_per90'] = pred_row.get('tackles_interceptions_per90', '')
            row['predicted_2025_assists'] = pred_row.get('assists_per_90', '')
            row['predicted_2025_progressive_passes'] = pred_row.get('progressive_passes_per90', '')
            row['predicted_2025_key_passes'] = pred_row.get('key_passes_per90', '')
            row['predicted_2025_tackles_interceptions'] = pred_row.get('tackles_interceptions_per90', '')

        row['explanation'] = pred_row.get('explanation', '')

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

# Save
output_path = '/content/drive/MyDrive/player_ratings/'
comparison_df.to_csv(f'{output_path}comparison_2024_vs_2025_predictions.csv',
                     index=False)

print(f"Total players: {len(comparison_df)}")
print(f"Columns: {len(comparison_df.columns)}")
print(f"\nSample ST comparison:")
st_cols = ['player', 'squad', 'subposition',
           'actual_2024_minutes', 'actual_2024_goals', 'actual_2024_assists',
           'predicted_2025_minutes', 'predicted_2025_goals', 'predicted_2025_assists',
           'explanation']
print(comparison_df[comparison_df['subposition'] == 'ST'][
    st_cols].head(5).to_string())

print(f"\nSample GK comparison:")
gk_cols = ['player', 'squad',
           'actual_2024_save_pct', 'predicted_2025_save_pct',
           'actual_2024_ga_per_90', 'predicted_2025_ga_per_90']
print(comparison_df[comparison_df['subposition'] == 'GK'][
    gk_cols].head(5).to_string())

print("\nComparison file saved! ✓")

BUILDING FINAL COMPARISON FILE
Total players: 393
Columns: 56

Sample ST comparison:
            player          squad subposition  actual_2024_minutes  actual_2024_goals  actual_2024_assists  predicted_2025_minutes  predicted_2025_goals  predicted_2025_assists                                                                                                                                 explanation
0   Alexander Isak  Newcastle Utd          ST                 2756               23.0                  6.0                    1890                  10.8                     3.1                                            At 25, entering peak years. Goal rate has declined recently. ↓ Playing more minutes each season.
1  Antoine Semenyo    Bournemouth          ST                 3203               11.0                  5.0                    2367                   7.6                     4.0                                              At 24, entering peak years. Goal rate has been consistent. 

In [ ]:
print("Checking for missing data in comparison file...")

# Check for NaN/empty values per column
print(f"\nTotal players: {len(comparison_df)}")
print(f"\nMissing values per column (top 15):")
print(comparison_df.isnull().sum().sort_values(ascending=False).head(15))

# Sample players with missing actual_2024 data
print("\nSample players missing actual 2024 data:")
if 'actual_2024_minutes' in comparison_df.columns:
    missing_actual = comparison_df[comparison_df['actual_2024_minutes'].isna()]
    print(f"Count: {len(missing_actual)}")
    print(missing_actual[['player', 'squad', 'subposition']].head(10).to_string())

# Sample players with missing predicted data
print("\nSample players missing predicted goals (ST/W):")
st_w = comparison_df[comparison_df['subposition'].isin(['ST', 'W'])]
if 'predicted_2025_goals' in st_w.columns:
    missing_pred = st_w[st_w['predicted_2025_goals'].isna() | (st_w['predicted_2025_goals'] == '')]
    print(f"Count: {len(missing_pred)}")
    print(missing_pred[['player', 'squad', 'subposition']].head(10).to_string())

Checking for missing data in comparison file...

Total players: 393

Missing values per column (top 15):
actual_2024_clean_sheet_pct                   393
actual_2024_save_pct                          393
actual_2024_ga_per_90                         393
predicted_2025_save_pct                       366
predicted_2025_ga_per_90                      366
predicted_2025_clean_sheet_pct                366
predicted_2025_key_passes_per90               348
predicted_2025_progressive_passes             348
predicted_2025_key_passes                     348
predicted_2025_progressive_passes_per90       348
predicted_2025_goals_low                      330
predicted_2025_goals_high                     330
predicted_2025_aerials_won_pct                321
predicted_2025_tackles_interceptions          311
predicted_2025_tackles_interceptions_per90    311
dtype: int64

Sample players missing actual 2024 data:
Count: 0
Empty DataFrame
Columns: [player, squad, subposition]
Index: []

Sample players m

In [ ]:
print("=" * 55)
print("CREATING POSITION-SPECIFIC COMPARISON FILES")
print("=" * 55)

output_path = '/content/drive/MyDrive/player_ratings/'

# Column rename mapping for readability
rename_map = {
    'player': 'Player',
    'squad': 'Team',
    'age': 'Age',
    'position': 'Position',
    'subposition': 'Role',
    'explanation': 'Analysis & Trend',

    'actual_2024_minutes': '2024 Minutes Played',
    'predicted_2025_minutes': '2025 Predicted Minutes',

    # ST/W
    'actual_2024_goals': '2024 Goals',
    'predicted_2025_goals': '2025 Predicted Goals',
    'predicted_2025_goals_low': '2025 Goals (Low Estimate)',
    'predicted_2025_goals_high': '2025 Goals (High Estimate)',
    'actual_2024_assists': '2024 Assists',
    'predicted_2025_assists': '2025 Predicted Assists',
    'actual_2024_xg': '2024 Expected Goals (xG)',
    'predicted_2025_xg': '2025 Predicted xG',
    'actual_2024_shots': '2024 Shots',
    'predicted_2025_shots': '2025 Predicted Shots',
    'actual_2024_goals_per90': '2024 Goals per 90',
    'predicted_2025_goals_per90': '2025 Predicted Goals per 90',
    'actual_2024_assists_per90': '2024 Assists per 90',
    'predicted_2025_assists_per90': '2025 Predicted Assists per 90',
    'actual_2024_xg_per90': '2024 xG per 90',
    'predicted_2025_xg_per90': '2025 Predicted xG per 90',
    'actual_2024_shots_per90': '2024 Shots per 90',
    'predicted_2025_shots_per90': '2025 Predicted Shots per 90',

    # CB/FB
    'actual_2024_aerials_won_pct': '2024 Aerial Duels Won %',
    'predicted_2025_aerials_won_pct': '2025 Predicted Aerial Duels Won %',
    'actual_2024_pass_completion_pct': '2024 Pass Completion %',
    'predicted_2025_pass_completion_pct': '2025 Predicted Pass Completion %',
    'actual_2024_interceptions': '2024 Interceptions',
    'predicted_2025_interceptions': '2025 Predicted Interceptions',
    'actual_2024_interceptions_per90': '2024 Interceptions per 90',
    'predicted_2025_interceptions_per90': '2025 Predicted Interceptions per 90',
    'actual_2024_clearances': '2024 Clearances',
    'predicted_2025_clearances': '2025 Predicted Clearances',
    'actual_2024_clearances_per90': '2024 Clearances per 90',
    'predicted_2025_clearances_per90': '2025 Predicted Clearances per 90',

    # DM/CM/AM
    'actual_2024_progressive_passes': '2024 Progressive Passes',
    'predicted_2025_progressive_passes': '2025 Predicted Progressive Passes',
    'actual_2024_progressive_passes_per90': '2024 Progressive Passes per 90',
    'predicted_2025_progressive_passes_per90': '2025 Predicted Progressive Passes per 90',
    'actual_2024_key_passes': '2024 Key Passes',
    'predicted_2025_key_passes': '2025 Predicted Key Passes',
    'actual_2024_key_passes_per90': '2024 Key Passes per 90',
    'predicted_2025_key_passes_per90': '2025 Predicted Key Passes per 90',
    'actual_2024_tackles_interceptions': '2024 Tackles + Interceptions',
    'predicted_2025_tackles_interceptions': '2025 Predicted Tackles + Interceptions',
    'actual_2024_tackles_interceptions_per90': '2024 Tackles + Interceptions per 90',
    'predicted_2025_tackles_interceptions_per90': '2025 Predicted Tackles + Interceptions per 90',

    # GK
    'actual_2024_save_pct': '2024 Save %',
    'predicted_2025_save_pct': '2025 Predicted Save %',
    'actual_2024_clean_sheet_pct': '2024 Clean Sheet %',
    'predicted_2025_clean_sheet_pct': '2025 Predicted Clean Sheet %',
    'actual_2024_ga_per_90': '2024 Goals Allowed per 90',
    'predicted_2025_ga_per_90': '2025 Predicted Goals Allowed per 90',
}

# Define column sets per role group
st_w_cols = ['player', 'squad', 'age', 'position', 'subposition',
              'actual_2024_minutes', 'predicted_2025_minutes',
              'actual_2024_goals', 'predicted_2025_goals',
              'predicted_2025_goals_low', 'predicted_2025_goals_high',
              'actual_2024_assists', 'predicted_2025_assists',
              'actual_2024_xg', 'predicted_2025_xg',
              'actual_2024_shots', 'predicted_2025_shots',
              'actual_2024_goals_per90', 'predicted_2025_goals_per90',
              'actual_2024_assists_per90', 'predicted_2025_assists_per90',
              'actual_2024_xg_per90', 'predicted_2025_xg_per90',
              'actual_2024_shots_per90', 'predicted_2025_shots_per90',
              'explanation']

cb_fb_cols = ['player', 'squad', 'age', 'position', 'subposition',
               'actual_2024_minutes', 'predicted_2025_minutes',
               'actual_2024_aerials_won_pct', 'predicted_2025_aerials_won_pct',
               'actual_2024_pass_completion_pct', 'predicted_2025_pass_completion_pct',
               'actual_2024_interceptions', 'predicted_2025_interceptions',
               'actual_2024_interceptions_per90', 'predicted_2025_interceptions_per90',
               'actual_2024_clearances', 'predicted_2025_clearances',
               'actual_2024_clearances_per90', 'predicted_2025_clearances_per90',
               'explanation']

dm_cm_am_cols = ['player', 'squad', 'age', 'position', 'subposition',
                  'actual_2024_minutes', 'predicted_2025_minutes',
                  'actual_2024_assists', 'predicted_2025_assists',
                  'actual_2024_assists_per90', 'predicted_2025_assists_per90',
                  'actual_2024_progressive_passes', 'predicted_2025_progressive_passes',
                  'actual_2024_progressive_passes_per90', 'predicted_2025_progressive_passes_per90',
                  'actual_2024_key_passes', 'predicted_2025_key_passes',
                  'actual_2024_key_passes_per90', 'predicted_2025_key_passes_per90',
                  'actual_2024_tackles_interceptions', 'predicted_2025_tackles_interceptions',
                  'actual_2024_tackles_interceptions_per90', 'predicted_2025_tackles_interceptions_per90',
                  'explanation']

gk_cols = ['player', 'squad', 'age', 'position', 'subposition',
            'actual_2024_minutes', 'predicted_2025_minutes',
            'actual_2024_save_pct', 'predicted_2025_save_pct',
            'actual_2024_clean_sheet_pct', 'predicted_2025_clean_sheet_pct',
            'actual_2024_ga_per_90', 'predicted_2025_ga_per_90',
            'explanation']

position_col_map = {
    'ST': st_w_cols, 'W': st_w_cols,
    'CB': cb_fb_cols, 'FB': cb_fb_cols,
    'DM': dm_cm_am_cols, 'CM': dm_cm_am_cols, 'AM': dm_cm_am_cols,
    'GK': gk_cols
}

# Friendly filenames
filename_map = {
    'ST': 'Strikers', 'W': 'Wingers',
    'CB': 'Centre_Backs', 'FB': 'Fullbacks',
    'DM': 'Defensive_Midfielders', 'CM': 'Central_Midfielders',
    'AM': 'Attacking_Midfielders', 'GK': 'Goalkeepers'
}

for subpos, cols in position_col_map.items():
    subset = comparison_df[comparison_df['subposition'] == subpos].copy()
    valid_cols = [c for c in cols if c in subset.columns]
    subset = subset[valid_cols]

    # Rename columns to readable names
    subset = subset.rename(columns=rename_map)

    # Sort by predicted main stat (goals for ST/W, etc.)
    sort_col = None
    for c in subset.columns:
        if 'Predicted Goals' in c and 'per 90' not in c and 'Low' not in c and 'High' not in c:
            sort_col = c
            break
    if sort_col:
        subset = subset.sort_values(sort_col, ascending=False)

    filename = f"{filename_map[subpos]}_2025_Predictions.csv"
    subset.to_csv(f'{output_path}{filename}', index=False)
    print(f"  {filename}: {len(subset)} players, {len(subset.columns)} columns")

print("\nAll position files saved with readable column names! ✓")

CREATING POSITION-SPECIFIC COMPARISON FILES
  Strikers_2025_Predictions.csv: 37 players, 26 columns
  Wingers_2025_Predictions.csv: 64 players, 26 columns
  Centre_Backs_2025_Predictions.csv: 72 players, 20 columns
  Fullbacks_2025_Predictions.csv: 82 players, 20 columns
  Defensive_Midfielders_2025_Predictions.csv: 66 players, 24 columns
  Central_Midfielders_2025_Predictions.csv: 16 players, 24 columns
  Attacking_Midfielders_2025_Predictions.csv: 29 players, 24 columns
  Goalkeepers_2025_Predictions.csv: 27 players, 14 columns

All position files saved with readable column names! ✓


In [ ]:
print("=" * 55)
print("CREATING PREDICTION-ONLY FILES")
print("=" * 55)

output_path = '/content/drive/MyDrive/player_ratings/'

# Prediction-only column sets per role group
st_w_pred_cols = ['player', 'squad', 'age', 'position', 'subposition',
                   'predicted_2025_minutes',
                   'predicted_2025_goals', 'predicted_2025_goals_low', 'predicted_2025_goals_high',
                   'predicted_2025_assists',
                   'predicted_2025_xg', 'predicted_2025_shots',
                   'predicted_2025_goals_per90', 'predicted_2025_assists_per90',
                   'predicted_2025_xg_per90', 'predicted_2025_shots_per90',
                   'explanation']

cb_fb_pred_cols = ['player', 'squad', 'age', 'position', 'subposition',
                     'predicted_2025_minutes',
                     'predicted_2025_aerials_won_pct', 'predicted_2025_pass_completion_pct',
                     'predicted_2025_interceptions', 'predicted_2025_interceptions_per90',
                     'predicted_2025_clearances', 'predicted_2025_clearances_per90',
                     'explanation']

dm_cm_am_pred_cols = ['player', 'squad', 'age', 'position', 'subposition',
                        'predicted_2025_minutes',
                        'predicted_2025_assists', 'predicted_2025_assists_per90',
                        'predicted_2025_progressive_passes', 'predicted_2025_progressive_passes_per90',
                        'predicted_2025_key_passes', 'predicted_2025_key_passes_per90',
                        'predicted_2025_tackles_interceptions', 'predicted_2025_tackles_interceptions_per90',
                        'explanation']

gk_pred_cols = ['player', 'squad', 'age', 'position', 'subposition',
                 'predicted_2025_minutes',
                 'predicted_2025_save_pct', 'predicted_2025_clean_sheet_pct',
                 'predicted_2025_ga_per_90',
                 'explanation']

position_pred_col_map = {
    'ST': st_w_pred_cols, 'W': st_w_pred_cols,
    'CB': cb_fb_pred_cols, 'FB': cb_fb_pred_cols,
    'DM': dm_cm_am_pred_cols, 'CM': dm_cm_am_pred_cols, 'AM': dm_cm_am_pred_cols,
    'GK': gk_pred_cols
}

filename_map = {
    'ST': 'Strikers', 'W': 'Wingers',
    'CB': 'Centre_Backs', 'FB': 'Fullbacks',
    'DM': 'Defensive_Midfielders', 'CM': 'Central_Midfielders',
    'AM': 'Attacking_Midfielders', 'GK': 'Goalkeepers'
}

for subpos, cols in position_pred_col_map.items():
    subset = comparison_df[comparison_df['subposition'] == subpos].copy()
    valid_cols = [c for c in cols if c in subset.columns]
    subset = subset[valid_cols]

    # Rename columns to readable names
    subset = subset.rename(columns=rename_map)

    # Sort by predicted main stat
    sort_col = None
    for c in subset.columns:
        if 'Predicted Goals' in c and 'per 90' not in c and 'Low' not in c and 'High' not in c:
            sort_col = c
            break
        elif 'Predicted Save' in c:
            sort_col = c
            break
        elif 'Predicted Aerial' in c:
            sort_col = c
            break
        elif 'Predicted Assists' in c and 'per 90' not in c:
            sort_col = c
            break
    if sort_col:
        subset = subset.sort_values(sort_col, ascending=False)

    filename = f"{filename_map[subpos]}_2025_Predictions_Only.csv"
    subset.to_csv(f'{output_path}{filename}', index=False)
    print(f"  {filename}: {len(subset)} players, {len(subset.columns)} columns")

print("\nAll prediction-only files saved! ✓")
print("\nYou now have 16 files total:")
print("  8 comparison files (Actual 2024 vs Predicted 2025)")
print("  8 prediction-only files (2025 predictions only)")

CREATING PREDICTION-ONLY FILES
  Strikers_2025_Predictions_Only.csv: 37 players, 17 columns
  Wingers_2025_Predictions_Only.csv: 64 players, 17 columns
  Centre_Backs_2025_Predictions_Only.csv: 72 players, 13 columns
  Fullbacks_2025_Predictions_Only.csv: 82 players, 13 columns
  Defensive_Midfielders_2025_Predictions_Only.csv: 66 players, 15 columns
  Central_Midfielders_2025_Predictions_Only.csv: 16 players, 15 columns
  Attacking_Midfielders_2025_Predictions_Only.csv: 29 players, 15 columns
  Goalkeepers_2025_Predictions_Only.csv: 27 players, 10 columns

All prediction-only files saved! ✓

You now have 16 files total:
  8 comparison files (Actual 2024 vs Predicted 2025)
  8 prediction-only files (2025 predictions only)


In [ ]:
print("Checking player coverage per team per position...")

for subpos in ['ST', 'W', 'CB', 'FB', 'DM', 'CM', 'AM', 'GK']:
    subset = comparison_df[comparison_df['subposition'] == subpos]
    team_counts = subset['squad'].value_counts()
    print(f"\n{subpos}: {len(subset)} total players across {subset['squad'].nunique()} teams")
    print(f"  Players per team: min={team_counts.min()}, max={team_counts.max()}, avg={team_counts.mean():.1f}")

    # Show teams with only 1 player
    one_player_teams = team_counts[team_counts == 1]
    if len(one_player_teams) > 0:
        print(f"  Teams with only 1 player: {len(one_player_teams)}")

Checking player coverage per team per position...

ST: 37 total players across 16 teams
  Players per team: min=1, max=5, avg=2.3
  Teams with only 1 player: 4

W: 64 total players across 23 teams
  Players per team: min=1, max=7, avg=2.8
  Teams with only 1 player: 6

CB: 72 total players across 23 teams
  Players per team: min=1, max=6, avg=3.1
  Teams with only 1 player: 6

FB: 82 total players across 20 teams
  Players per team: min=1, max=9, avg=4.1
  Teams with only 1 player: 1

DM: 66 total players across 21 teams
  Players per team: min=1, max=7, avg=3.1
  Teams with only 1 player: 3

CM: 16 total players across 11 teams
  Players per team: min=1, max=3, avg=1.5
  Teams with only 1 player: 7

AM: 29 total players across 17 teams
  Players per team: min=1, max=4, avg=1.7
  Teams with only 1 player: 10

GK: 27 total players across 20 teams
  Players per team: min=1, max=2, avg=1.4
  Teams with only 1 player: 13


In [ ]:
print("Checking total player coverage...")

# Total unique players in 2024 (before our 5x90 filter)
raw_2024 = pd.read_csv('/content/drive/MyDrive/player_ratings/data/2024_PL.csv')
print(f"Total players in raw 2024 file: {len(raw_2024)}")

# After filtering to 5+ x90s
filtered_2024 = raw_2024[raw_2024['ninety_mins_played'] >= 5]
print(f"Players with 5+ x90s (450+ mins): {len(filtered_2024)}")

# Players below 5 x90s (excluded)
excluded = raw_2024[raw_2024['ninety_mins_played'] < 5]
print(f"Players excluded (< 5 x90s / 450 mins): {len(excluded)}")

# GK count
gk_2024_raw = pd.read_csv('/content/drive/MyDrive/player_ratings/data/GK_2024_PL.csv')
print(f"\nTotal GKs in raw 2024 GK file: {len(gk_2024_raw)}")
gk_filtered = gk_2024_raw[gk_2024_raw['ninety_mins_played'] >= 5]
print(f"GKs with 5+ x90s: {len(gk_filtered)}")

print(f"\n{'='*40}")
print(f"Our predictions cover: 393 players")
print(f"Total raw players (outfield + GK): {len(raw_2024) + len(gk_2024_raw)}")
print(f"Excluded (low minutes): {len(excluded)}")

Checking total player coverage...
Total players in raw 2024 file: 611
Players with 5+ x90s (450+ mins): 366
Players excluded (< 5 x90s / 450 mins): 245

Total GKs in raw 2024 GK file: 33
GKs with 5+ x90s: 27

Our predictions cover: 393 players
Total raw players (outfield + GK): 644
Excluded (low minutes): 245
